# CE 639: AI for Civil Engineering
## Lecture 10: Feedforward Neural Networks (FNNs)

**Learning Objectives:**
1. Understand why nonlinear function approximators are needed for CE problems
2. Describe feedforward neural network architecture (layers, weights, activations)
3. Implement the forward pass from scratch in NumPy
4. Demonstrate weight initialisation, vanishing gradients, and regularisation
5. Apply FNNs to three realistic civil engineering problems
6. Identify when to use (and when NOT to use) neural networks

**Topics Covered:**
- Motivation: from linear models to nonlinear CE physics
- RC Beam running example (piecewise cracking physics)
- Universal Approximation Theorem (demonstrated interactively)
- The Perceptron — single neuron forward pass
- Network architecture: layers, depth, width, notation
- Why nonlinearity? — the linear collapse proof
- Activation functions (sigmoid, tanh, ReLU, Leaky ReLU, ELU, GELU)
- The forward pass: vectorised implementation
- Loss functions and training overview
- Weight initialisation: Xavier vs He
- Vanishing / exploding gradients
- Regularisation (L2, Dropout) and Batch Normalisation
- CE Applications: Beam deflection, Concrete strength, Traffic flow
- Depth × Width hyperparameter study
- Edge case stress testing

---
*CE 639: AI for Civil Engineering — IIT Gandhinagar*
*Instructors: Dr. Udit Bhatia & Dr. Sushobhan Sen*

---
## 1. Setup & Imports

We load our custom `utils/Lecture_10/` package (from-scratch NumPy implementations)
plus standard scientific Python libraries and PyTorch.

In [ ]:
# ─── Standard scientific stack ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# ─── Plotting style ───────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
np.random.seed(42)

print("✅ Core scientific packages loaded")

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab ✓")
except:
    IN_COLAB = False
    print("Running locally ✓")

if IN_COLAB:
    # !git clone https://github.com/dubeysarth/2026_IITGN_CE639
    # %cd 2026_IITGN_CE639
    import os
    subprocess.run(['git', 'clone', 'https://github.com/dubeysarth/2026_IITGN_CE639'])
    os.chdir('2026_IITGN_CE639')

In [ ]:
# ─── Our Lecture 10 utilities ─────────────────────────────────────────────────
import sys
sys.path.append('..')

try:
    from utils.Lecture_10 import (
        # Perceptron / single neuron
        perceptron_forward, perceptron_decision_boundary,
        single_neuron_gradient_step, single_neuron_gradient_descent,
        perceptron_train, relu_piecewise_boundary,
        # Activation functions
        sigmoid, sigmoid_derivative, tanh_act, tanh_derivative,
        relu, relu_derivative, leaky_relu, gelu,
        softmax, get_activation, get_derivative,
        activation_summary_table,
        compute_saturation_fraction, compute_dead_relu_fraction,
        # NumPy FNN
        NumpyFNN, init_weights, forward_pass_step_by_step,
        count_parameters, demonstrate_linear_collapse,
        # Training
        mse_loss, binary_cross_entropy_loss,
        train_numpy_fnn, train_pytorch_fnn, evaluate_pytorch,
        plot_training_history, plot_decision_regions,
        # PyTorch architectures
        SimpleFNN, BeamDeflectionNet, ConcreteStrengthNet,
        TrafficFlowNet, build_fnn, count_torch_parameters,
        # CE Datasets
        generate_beam_deflection_dataset,
        generate_concrete_strength_dataset,
        generate_traffic_flow_dataset,
        generate_xor_dataset, generate_spiral_dataset,
        generate_regression_1d,
        # Visualisations
        plot_activation_gallery, plot_network_diagram,
        plot_gradient_flow, plot_init_comparison,
        plot_regularization_comparison, plot_loss_landscape_2d,
        plot_depth_vs_width, plot_batch_norm_effect,
        # Interactive widgets
        activation_explorer_widget, forward_pass_widget,
        network_builder_widget, training_playground_widget,
        initialization_widget, learning_rate_widget,
    )
    print("✅ Lecture 10 utilities loaded successfully!")

except ImportError as e:
    print(f"⚠️  Import error: {e}")
    print("   Make sure utils/Lecture_10/ folder exists and all files are present.")

In [ ]:
# ─── Check PyTorch availability ───────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
    print(f"✅ PyTorch {torch.__version__} available")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"   Using device: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not available — NumPy implementations will be used instead.")
    print("   Install: pip install torch")

---
## 2. Motivation: From Linear Models to Nonlinear CE Physics

### Why Do We Need Neural Networks at All?

Recall from Lecture 6 (Regression): linear models fit hyperplanes through data.
They're interpretable, robust, and often good enough.

**But what if the physics is fundamentally nonlinear?**

Let's start with the classic toy problem — XOR — which is the simplest
example of a relationship that cannot be captured by any linear model,
then connect it to a real CE problem.

In [ ]:
# ─── XOR — the classic failure case of linear models ─────────────────────────
print("=" * 70)
print("XOR CLASSIFICATION: WHY LINEAR MODELS FAIL")
print("=" * 70)

X_xor, y_xor = generate_xor_dataset(n=400, noise=0.1, random_state=42)

print(f"\n📊 XOR Dataset:")
print(f"   Points: {X_xor.shape[0]}")
print(f"   Class balance: {np.bincount(y_xor)}")

# Try a linear classifier (logistic regression) manually
# Fit the best linear boundary on this data
from numpy.linalg import lstsq

# Augment with bias
X_aug = np.column_stack([X_xor, np.ones(len(X_xor))])
# Pseudo-OLS classification
w_linear, _, _, _ = lstsq(X_aug, y_xor, rcond=None)
y_lin_pred = (X_aug @ w_linear >= 0.5).astype(int)
lin_acc = np.mean(y_lin_pred == y_xor)

print(f"\n🔵 Linear Classifier Accuracy on XOR: {lin_acc:.1%}")
print(f"   (Random chance = 50%) → Linear model is USELESS here!")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# XOR data
colors = ['#2196F3', '#FF5722']
for cls, col in enumerate(colors):
    mask = y_xor == cls
    axes[0].scatter(X_xor[mask, 0], X_xor[mask, 1], c=col, s=40,
                    label=f"Class {cls}", edgecolors='white', linewidths=0.5, alpha=0.8)

# Best linear boundary
x1_line = np.linspace(X_xor[:, 0].min(), X_xor[:, 0].max(), 100)
if abs(w_linear[1]) > 1e-10:
    x2_line = -(w_linear[0] * x1_line + w_linear[2] - 0.5) / w_linear[1]
    axes[0].plot(x1_line, x2_line, 'k--', linewidth=2, label=f'Best linear boundary\n(Acc={lin_acc:.1%})')

axes[0].set_title('XOR Problem — Linear Model Fails', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x₁', fontsize=12)
axes[0].set_ylabel('x₂', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_xlim(-1.5, 1.5)
axes[0].set_ylim(-1.5, 1.5)

# Spiral — even harder
X_sp, y_sp = generate_spiral_dataset(n=200, noise=0.05, n_classes=2)
sp_colors = ['#2196F3', '#FF5722']
for cls, col in enumerate(sp_colors):
    mask = y_sp == cls
    axes[1].scatter(X_sp[mask, 0], X_sp[mask, 1], c=col, s=40,
                    label=f"Class {cls}", edgecolors='white', linewidths=0.5, alpha=0.8)
axes[1].set_title('Spiral Problem — Impossible for Linear Models', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x₁', fontsize=12)
axes[1].set_ylabel('x₂', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('Nonlinear Problems That Break Linear Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print("   Both XOR and the spiral have decision boundaries that NO straight line can separate.")
print("   We need a NONLINEAR function approximator → Feedforward Neural Networks!")

---
## 3. The RC Beam Running Example

### Physics Baseline — Elastic Beam Theory

The slides use RC beam midspan deflection as the running CE example.
Let's build the dataset and understand WHY a linear model is insufficient.

$$\\delta = \\frac{P L^3}{48 E I_{\\text{eff}}}$$

The key complication: $I_{\\text{eff}}$ is **not constant** — it changes at cracking!

In [ ]:
# ─── Generate beam dataset and explore the physics ────────────────────────────
print("=" * 70)
print("RC BEAM DEFLECTION — THE CE RUNNING EXAMPLE")
print("=" * 70)

X_beam, y_beam, beam_meta = generate_beam_deflection_dataset(n=1500, noise_frac=0.05)

print(f"\n📊 Beam Deflection Dataset:")
print(f"   Samples: {X_beam.shape[0]}")
print(f"   Features: {beam_meta['feature_names']}")
print(f"   Target: {beam_meta['target_name']}")
print(f"   Deflection range: [{y_beam.min():.1f}, {y_beam.max():.1f}] mm")
print(f"\n   Physics formula: {beam_meta['formula']}")

# Unpack features
P       = X_beam[:, 0]   # kN
L       = X_beam[:, 1]   # m
M_cr    = X_beam[:, 4]   # kN·m
M_max   = P * L / 4.0    # kN·m (simply supported, midspan point load)

# Show physics: cracking regime indicator
cracked_mask = M_max > M_cr

print(f"\n⚡ Regime Analysis:")
print(f"   Uncracked beams (M_max ≤ M_cr): {(~cracked_mask).sum()} ({(~cracked_mask).mean():.1%})")
print(f"   Cracked beams   (M_max  > M_cr): {cracked_mask.sum()} ({cracked_mask.mean():.1%})")
print(f"   → Piecewise behaviour: TWO different physics regimes!")

# Visualise: deflection vs load, colour-coded by cracking
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(P[~cracked_mask], y_beam[~cracked_mask],
                     c='steelblue', s=15, alpha=0.5, label='Uncracked (I = I_gross)')
sc = axes[0].scatter(P[cracked_mask], y_beam[cracked_mask],
                     c='tomato', s=15, alpha=0.5, label='Cracked (I < I_gross)')

axes[0].set_xlabel('Applied Load P (kN)', fontsize=12)
axes[0].set_ylabel('Midspan Deflection δ (mm)', fontsize=12)
axes[0].set_title('Load vs Deflection — Two Physics Regimes', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Deflection distribution
axes[1].hist(y_beam[~cracked_mask], bins=40, alpha=0.7, color='steelblue',
             label='Uncracked', density=True)
axes[1].hist(y_beam[cracked_mask], bins=40, alpha=0.7, color='tomato',
             label='Cracked', density=True)
axes[1].set_xlabel('Deflection (mm)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Deflection Distributions by Regime', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('RC Beam Dataset — Piecewise Physics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Key takeaway: No single smooth formula works for both regimes.")
print("   An FNN can learn the piecewise mapping: (P, L, E, I, M_cr, ρ) → δ")

In [ ]:
# ─── Why a linear model fails on the beam dataset ────────────────────────────
print("\n" + "=" * 50)
print("LINEAR MODEL BENCHMARK ON BEAM DATA")
print("=" * 50)

# Normalise
from sklearn.preprocessing import StandardScaler
scaler_beam = StandardScaler()
X_beam_scaled = scaler_beam.fit_transform(X_beam)

# Simple linear regression fit
X_aug_beam = np.column_stack([X_beam_scaled, np.ones(len(X_beam_scaled))])
w_beam, _, _, _ = lstsq(X_aug_beam, y_beam, rcond=None)
y_beam_lin = X_aug_beam @ w_beam

lin_mse = mse_loss(y_beam, y_beam_lin)
lin_r2 = 1 - np.sum((y_beam - y_beam_lin) ** 2) / np.sum((y_beam - y_beam.mean()) ** 2)

print(f"\n📉 Linear Model Performance:")
print(f"   MSE:   {lin_mse:.2f} mm²")
print(f"   RMSE:  {np.sqrt(lin_mse):.2f} mm")
print(f"   R²:    {lin_r2:.4f}  (1.0 = perfect)")
print(f"\n   → Linear model only explains {lin_r2:.1%} of variance.")
print(f"   → FNN will do much better by capturing the nonlinear cracking physics!")

---
## 4. Universal Approximation Theorem

**Theorem (informal):** A feedforward neural network with a *single* hidden layer
containing a finite number of neurons can approximate any continuous function
to arbitrary precision on a compact subset of ℝⁿ.

**Implication for CE:** Any smooth engineering input-output mapping
(load → deflection, weather → traffic) can theoretically be learned from data.

**Caveat:** The theorem says NOTHING about:
- How many neurons are needed
- How to find the weights (training may be hard)
- Generalisation to unseen data

Let's demonstrate this visually on 1-D functions.

In [ ]:
# ─── UAT Demo: approximate 1-D functions with increasing hidden neurons ────────
print("=" * 70)
print("UNIVERSAL APPROXIMATION THEOREM — INTERACTIVE DEMO")
print("=" * 70)

# Use a simple NumpyFNN to fit each function with increasing capacity
from sklearn.preprocessing import StandardScaler

functions = ["sin", "step", "runge"]
n_neurons_list = [2, 5, 10, 50]

fig, axes = plt.subplots(len(functions), len(n_neurons_list) + 1,
                         figsize=(18, 10))

x_range = (-3.0, 3.0)
X_1d, _ = generate_regression_1d("sin", n=200, noise=0, x_range=x_range)
x_plot = np.linspace(x_range[0], x_range[1], 300)

for i, fn_name in enumerate(functions):
    X_1d, y_1d = generate_regression_1d(fn_name, n=300, noise=0.05, x_range=x_range)
    y_true = get_activation("linear")(x_plot) if fn_name == "linear" else \
             np.vectorize(lambda x: __import__('utils.Lecture_10.ce_datasets',
                         fromlist=['generate_regression_1d'])
                         )(x_plot)

    # True function (no noise)
    X_true, y_true_clean = generate_regression_1d(fn_name, n=300, noise=0, x_range=x_range)

    # Plot ground truth
    ax = axes[i, 0]
    ax.scatter(X_1d.ravel(), y_1d, s=8, alpha=0.4, color='gray', label='Noisy data')
    ax.plot(X_true.ravel(), y_true_clean, 'k-', linewidth=2, label='True function')
    ax.set_title(f'True: {fn_name}', fontsize=10, fontweight='bold')
    ax.set_xlim(x_range)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

    # Fit NumpyFNN with increasing hidden neurons
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    Xs = scaler_x.fit_transform(X_1d)
    ys = scaler_y.fit_transform(y_1d.reshape(-1, 1)).ravel()

    for j, n_h in enumerate(n_neurons_list):
        model_uat = NumpyFNN([1, n_h, 1], activations=['relu', 'linear'],
                              init_method='he', random_state=j)
        history_uat = train_numpy_fnn(model_uat, Xs, ys,
                                       lr=0.01, epochs=500,
                                       batch_size=32, verbose=0)

        # Predict on dense grid
        x_pred_s = scaler_x.transform(x_plot.reshape(-1, 1))
        y_pred_s = model_uat.predict(x_pred_s).ravel()
        y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

        ax = axes[i, j + 1]
        ax.plot(X_true.ravel(), y_true_clean, 'k-', linewidth=1.5, alpha=0.6, label='True')
        ax.plot(x_plot, y_pred, 'tomato', linewidth=2, label=f'FNN ({n_h} neurons)')
        ax.scatter(X_1d.ravel(), y_1d, s=5, alpha=0.3, color='gray')
        final_loss = history_uat['train_loss'][-1]
        ax.set_title(f'N={n_h}, Loss={final_loss:.3f}', fontsize=10, fontweight='bold')
        ax.set_xlim(x_range)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.2)

col_labels = ['Ground Truth'] + [f'{n} Neurons' for n in n_neurons_list]
for j, label in enumerate(col_labels):
    axes[0, j].set_title(f'{label}\n{axes[0,j].get_title()}', fontsize=10, fontweight='bold')

row_labels = ['sin(2πx)', 'Step function', "Runge's function"]
for i, label in enumerate(row_labels):
    axes[i, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.suptitle('Universal Approximation: More Neurons → Better Fit',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 Key Observations from the UAT Demo:")
print("   • 2 neurons: captures only the coarsest trend")
print("   • 10 neurons: good fit for smooth functions (sin)")
print("   • 50 neurons: can approximate even sharp discontinuities (step)")
print("   • More neurons is not always better → risk of overfitting!")
print("   • Runge's function shows the effect of oscillatory approximation")

---
## 5. The Perceptron — Building Block of Neural Networks

The **perceptron** is the single-neuron unit:

$$z = \\mathbf{w}^T \\mathbf{x} + b$$
$$a = g(z)$$

where:
- $\\mathbf{w}$ = weight vector  
- $b$ = bias  
- $g$ = activation function

A single perceptron with **sigmoid** activation is **exactly logistic regression**
(from Lecture 7)!

In [ ]:
# ─── Single perceptron forward pass (Practice Problem 1 setup) ─────────────────
print("=" * 70)
print("THE PERCEPTRON — SINGLE NEURON")
print("=" * 70)

# Exact example from the slides (Practice Problem 1)
print("\n📝 Slide Practice Problem 1 — Forward Pass:")
print("   Input: x = [1, 2]")
print("   W^[1] = [[1, -1], [0.5, 2]], b^[1] = [0, -1]")

x_sample = np.array([1.0, 2.0])

# Single neurons for each row of W^[1]
for i, (w, b, label) in enumerate([
    (np.array([1.0, -1.0]), 0.0, "Neuron 1 (ReLU)"),
    (np.array([0.5, 2.0]), -1.0, "Neuron 2 (ReLU)"),
]):
    z, a = perceptron_forward(x_sample, w, b, activation='relu')
    print(f"\n   {label}:")
    print(f"     z = {w} · {x_sample} + {b} = {z:.2f}")
    print(f"     a = ReLU({z:.2f}) = {a:.2f}")

# Full layer via NumpyFNN
model_demo = NumpyFNN([2, 2, 1],
                       activations=['relu', 'linear'],
                       init_method='zero')
# Manually set weights exactly as in the slides
model_demo.W[0] = np.array([[1.0, -1.0], [0.5, 2.0]])
model_demo.b[0] = np.array([0.0, -1.0])
model_demo.W[1] = np.array([[2.0, -1.0]])
model_demo.b[1] = np.array([0.5])

out_demo = model_demo.forward(x_sample.reshape(1, -1))
a1 = model_demo.cache['a1'].ravel()

print(f"\n   Full Layer 1 output: a^[1] = {a1}")
print(f"   W^[2] = [2, -1], b^[2] = 0.5")
print(f"   ŷ = W^[2] · a^[1] + b^[2] = {out_demo.ravel()[0]:.2f}")
print(f"\n   (Slides say ŷ = -3.0 ✓ — slight difference from exact problem values)")

In [ ]:
# ─── Decision boundary for a single neuron ────────────────────────────────────
print("\n" + "=" * 50)
print("PERCEPTRON DECISION BOUNDARY (2-D)")
print("=" * 50)

# Practice Problem 2 from the slides: w=[2, -3], b=1
w_p2 = np.array([2.0, -3.0])
b_p2 = 1.0

boundary = relu_piecewise_boundary(w_p2, b_p2)
print(f"\n📝 Slide Practice Problem 2:")
print(f"   w = {w_p2}, b = {b_p2}")
print(f"   Boundary: {boundary['boundary_eq']}")
print(f"   x₂ form: {boundary.get('x2_formula', 'N/A')}")

# Compute activations for the three test points
test_points = [(1.0, 0.0), (0.0, 1.0), (2.0, 1.0)]
print("\n   Activations at test points:")
for x1, x2 in test_points:
    x_test = np.array([x1, x2])
    z_test, a_test = perceptron_forward(x_test, w_p2, b_p2, 'relu')
    print(f"   x={x_test}: z = {z_test:.1f}, a = ReLU({z_test:.1f}) = {a_test:.1f}")

# Visualise
from utils.Lecture_10.perceptron import neuron_activation_region
X1, X2, Z = neuron_activation_region(w_p2, b_p2, grid_size=300, x_range=(-2, 3))
x1_bound, x2_bound = perceptron_decision_boundary(w_p2, b_p2, x_range=(-2, 3))

fig, ax = plt.subplots(figsize=(8, 7))
cf = ax.contourf(X1, X2, Z, levels=20, cmap='RdYlGn', alpha=0.6)
plt.colorbar(cf, ax=ax, label='z = w·x + b (pre-activation)')
ax.contour(X1, X2, Z, levels=[0], colors='black', linewidths=2.5, linestyles='--')
ax.plot(x1_bound, x2_bound, 'k--', linewidth=2.5, label='Boundary: z=0 (ReLU switches)')
ax.axhline(0, color='gray', linewidth=0.7)
ax.axvline(0, color='gray', linewidth=0.7)

for (x1, x2) in test_points:
    z_v, a_v = perceptron_forward(np.array([x1, x2]), w_p2, b_p2, 'relu')
    color = 'tomato' if a_v > 0 else 'steelblue'
    ax.scatter(x1, x2, c=color, s=200, zorder=5, edgecolors='black', linewidths=2)
    ax.text(x1 + 0.05, x2 + 0.1, f'a={a_v:.1f}', fontsize=11)

ax.set_xlabel('x₁', fontsize=12)
ax.set_ylabel('x₂', fontsize=12)
ax.set_title('Single ReLU Neuron — Piecewise Linear Boundary\n'
             'Green = ON (z>0), Red = OFF (z≤0)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-2, 3)
ax.set_ylim(-2, 3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Insight: The boundary where a ReLU neuron switches")
print("   ON/OFF is a hyperplane in input space — always linear!")
print("   Neural networks create complex NONLINEAR boundaries by combining many such hyperplanes.")

### Practice Problem 2 — Solved Interactively

Use the widget below to drag `w` and `b` and watch the decision boundary move!

In [ ]:
# Interactive: Perceptron decision boundary explorer
# Uncomment in Jupyter/Colab:
# forward_pass_widget()

print("🎮 Interactive Forward Pass Widget")
print("=" * 50)
print("Run in Jupyter/Colab:  forward_pass_widget()")
print("\nControls:")
print("  • x₁, x₂  — Input values")
print("  • W[i,j]   — Weight matrix entries")
print("  • b₁, b₂   — Bias values")
print("  • Activation — relu, sigmoid, tanh, etc.")
print("\nSee live pre-activation z and post-activation a for each neuron!")

---
## 6. Layered Architecture & Mathematical Notation

A **feedforward neural network** stacks multiple layers:

$$\\mathbf{a}^{[0]} = \\mathbf{x}  \\quad (\\text{input})$$

For each layer $l = 1, 2, \\ldots, L$:
$$\\mathbf{z}^{[l]} = \\mathbf{W}^{[l]} \\mathbf{a}^{[l-1]} + \\mathbf{b}^{[l]}$$
$$\\mathbf{a}^{[l]} = g^{[l]}(\\mathbf{z}^{[l]})$$

Final output: $\\hat{\\mathbf{y}} = \\mathbf{a}^{[L]}$

**Dimensions (slides §8):**
- $\\mathbf{W}^{[l]} \\in \\mathbb{R}^{n_l \\times n_{l-1}}$ — connects layer $l-1$ to layer $l$
- $\\mathbf{b}^{[l]} \\in \\mathbb{R}^{n_l}$
- $\\mathbf{a}^{[l]} \\in \\mathbb{R}^{n_l}$

In [ ]:
# ─── Visualise the network architecture ─────────────────────────────────────────
print("=" * 70)
print("FNN ARCHITECTURE — DIMENSION TRACKING")
print("=" * 70)

# Our beam deflection network: 6 inputs → 2 hidden → 1 output
layer_sizes_beam = [6, 32, 16, 1]
activations_beam = ["ReLU", "ReLU", "Linear"]

param_info = count_parameters(layer_sizes_beam)

print(f"\n🏗️  Beam Deflection FNN Architecture:")
print(f"   Layer sizes: {layer_sizes_beam}")
print(f"\n   {'Layer':<8} {'W shape':<18} {'b shape':<12} {'Activation':<12} {'Params':>8}")
print(f"   {'-'*60}")
for p in param_info['per_layer']:
    act = activations_beam[p['layer']-1] if p['layer'] <= len(activations_beam) else "?"
    print(f"   L{p['layer']:<7} ({p['n_out']}×{p['n_in']}){'':<6} ({p['n_out']},){'':<4} {act:<12} {p['params']:>8,}")
print(f"   {'-'*60}")
print(f"   {'TOTAL':<48} {param_info['total']:>8,}")

# Draw the diagram
fig, ax = plot_network_diagram(
    layer_sizes_beam,
    activations=["ReLU", "ReLU", "Linear"],
    title=f"Beam Deflection FNN: {layer_sizes_beam}  ({param_info['total']:,} parameters)"
)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Interactive network builder ─────────────────────────────────────────────
# Uncomment in Jupyter/Colab:
# network_builder_widget()

print("🎮 Interactive Network Builder Widget")
print("=" * 50)
print("Run in Jupyter/Colab:  network_builder_widget()")
print("\nControls:")
print("  • Input dim, # hidden layers, neurons/layer, output dim")
print("  • Activation function")
print("\nSee parameter count and network diagram update in real time!")

---
## 7. Why Nonlinearity? — The Linear Collapse Proof

What happens if we stack many linear layers **without** any activation function?

Let's prove algebraically and numerically that the network collapses to a
**single linear transformation** regardless of depth!

$$\\mathbf{z}^{[2]} = \\mathbf{W}^{[2]} (\\mathbf{W}^{[1]} \\mathbf{x} + \\mathbf{b}^{[1]}) + \\mathbf{b}^{[2]}$$
$$= (\\mathbf{W}^{[2]} \\mathbf{W}^{[1]}) \\mathbf{x} + (\\mathbf{W}^{[2]} \\mathbf{b}^{[1]} + \\mathbf{b}^{[2]})$$
$$= \\underbrace{\\mathbf{W}_{\\text{eff}}}_{\\text{single matrix}} \\mathbf{x} + \\underbrace{\\mathbf{b}_{\\text{eff}}}_{\\text{single bias}}$$

**Depth is wasted without nonlinear activations!**

In [ ]:
# ─── Linear collapse proof ────────────────────────────────────────────────────
print("=" * 70)
print("LINEAR COLLAPSE PROOF — NUMERICAL DEMONSTRATION")
print("=" * 70)

for depth, layer_sizes in [(2, [4, 8, 1]), (5, [4, 8, 8, 8, 8, 1])]:
    result = demonstrate_linear_collapse(layer_sizes)
    print(f"\n🔢 {depth}-layer network {layer_sizes}:")
    print(f"   Product W^[L]...W^[1] shape: {result['W_product_shape']}")
    print(f"   Rank = {result['rank']} (max possible = {result['max_possible_rank']})")
    print(f"   → Any depth collapses to shape {result['W_product_shape']}!")

# Visualise: nonlinear vs linear network on XOR
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Linear 3-layer network on XOR (should fail)
model_linear = NumpyFNN([2, 16, 16, 1],
                         activations=['linear', 'linear', 'linear'],
                         init_method='he')
X_xor_scaled = X_xor / (X_xor.std() + 1e-8)
hist_lin_xor = train_numpy_fnn(model_linear, X_xor_scaled, y_xor.astype(float),
                                lr=0.01, epochs=200, batch_size=32,
                                loss_fn='bce', verbose=0)

# Nonlinear 3-layer network on XOR (should succeed)
model_relu = NumpyFNN([2, 16, 16, 1],
                       activations=['relu', 'relu', 'sigmoid'],
                       init_method='he')
hist_relu_xor = train_numpy_fnn(model_relu, X_xor_scaled, y_xor.astype(float),
                                 lr=0.01, epochs=200, batch_size=32,
                                 loss_fn='bce', verbose=0)

# Decision regions
def linear_model_fn(X):
    Xs = X / (X.std() + 1e-8)
    return model_linear.predict(Xs)

def relu_model_fn(X):
    Xs = X / (X.std() + 1e-8)
    return model_relu.predict(Xs)

for ax_idx, (model_fn, title, hist) in enumerate([
    (linear_model_fn, "Linear Activations\n(3-layer, collapses!)", hist_lin_xor),
    (relu_model_fn,   "ReLU Activations\n(3-layer, works!)", hist_relu_xor),
]):
    ax = axes[ax_idx]
    x1_range = np.linspace(X_xor[:, 0].min() - 0.5, X_xor[:, 0].max() + 0.5, 200)
    x2_range = np.linspace(X_xor[:, 1].min() - 0.5, X_xor[:, 1].max() + 0.5, 200)
    XX1, XX2 = np.meshgrid(x1_range, x2_range)
    grid = np.c_[XX1.ravel(), XX2.ravel()]
    Z_grid = model_fn(grid).reshape(XX1.shape)

    ax.contourf(XX1, XX2, Z_grid, levels=20, cmap='RdYlBu', alpha=0.5)
    for cls, col in enumerate(['#2196F3', '#FF5722']):
        mask = y_xor == cls
        ax.scatter(X_xor[mask, 0], X_xor[mask, 1], c=col, s=30, edgecolors='white',
                   linewidths=0.5, alpha=0.9, label=f'Class {cls}')
    final_acc = np.mean((model_fn(X_xor_scaled) >= 0.5).astype(int).ravel() == y_xor)
    ax.set_title(f'{title}\nFinal Acc: {final_acc:.1%}', fontsize=11, fontweight='bold')
    ax.set_xlabel('x₁', fontsize=11)
    ax.set_ylabel('x₂', fontsize=11)
    ax.legend(fontsize=9)

# Training curves comparison
axes[2].plot(hist_lin_xor['train_loss'], 'tomato', linewidth=2, label='Linear (collapses)')
axes[2].plot(hist_relu_xor['train_loss'], 'steelblue', linewidth=2, label='ReLU (learns)')
axes[2].set_xlabel('Epoch', fontsize=11)
axes[2].set_ylabel('Loss', fontsize=11)
axes[2].set_title('Training Loss: Linear vs ReLU\non XOR problem', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Without Nonlinearity, Neural Networks Are Just Linear Models',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Conclusion: Without activation functions, stacking layers = waste of computation.")
print("   Nonlinear activations are the ENGINE of neural networks!")

---
## 8. Activation Functions

The choice of activation function has major practical implications:
- **Sigmoid / Tanh**: saturate for large |z| → vanishing gradients (historical issue)
- **ReLU**: no saturation for z > 0, fast, sparse, **DEFAULT CHOICE**
- **Leaky ReLU**: fixes dead neurons
- **ELU**: smooth, near-zero mean activations
- **GELU**: used in transformers (smooth approximation)

For the **output layer**, the activation depends on the task:
- Regression → **Linear** (no activation)
- Binary classification → **Sigmoid**
- Multi-class classification → **Softmax**

In [ ]:
# ─── Activation function gallery ─────────────────────────────────────────────
print("=" * 70)
print("ACTIVATION FUNCTIONS — GALLERY")
print("=" * 70)

fig, axes = plot_activation_gallery(z_range=(-5.0, 5.0), figsize=(16, 10))
plt.tight_layout()
plt.show()

In [ ]:
# ─── Print the summary table ───────────────────────────────────────────────────
print("\n📋 Activation Function Quick Reference:")
print(f"{'Name':<16} {'Formula':<40} {'Range':<12} {'Use For'}")
print("-" * 95)
table = activation_summary_table()
for name, info in table.items():
    print(f"{name:<16} {info['formula']:<40} {info['range']:<12} {info['use_for']}")

In [ ]:
# ─── Saturation / dead-neuron demonstration ─────────────────────────────────────
print("\n" + "=" * 55)
print("SATURATION & DEAD NEURON DEMONSTRATION")
print("=" * 55)

# Large-magnitude inputs (simulating poor normalisation)
z_poor = np.concatenate([np.random.randn(500) * 5, np.random.randn(500) * 0.5 - 3])

for act_name in ['sigmoid', 'tanh', 'relu', 'leaky_relu']:
    sat = compute_saturation_fraction(z_poor, act_name, threshold=0.05)
    if act_name == 'relu':
        dead = compute_dead_relu_fraction(z_poor)
        print(f"  {act_name:<15} Saturated: {sat:.1%}   Dead neurons: {dead:.1%}")
    else:
        print(f"  {act_name:<15} Saturated: {sat:.1%}")

print(f"\n💡 Key Observations:")
print(f"  • Sigmoid & Tanh: MANY saturated neurons for |z| >> 0 → gradient ≈ 0!")
print(f"  • ReLU: Dead neurons for z ≤ 0, but no saturation for z > 0")
print(f"  • Leaky ReLU: No dead neurons, no saturation → best of both worlds")
print(f"\n  Solution for sigmoid/tanh saturation: proper weight initialisation + normalised inputs")

In [ ]:
# ─── Interactive activation explorer ─────────────────────────────────────────
# Uncomment in Jupyter/Colab:
# activation_explorer_widget()

print("\n🎮 Interactive Activation Explorer")
print("=" * 50)
print("Run in Jupyter/Colab:  activation_explorer_widget()")
print("\nAdjust:")
print("  • Activation type (dropdown)")
print("  • Z range (slider)")
print("  • Toggle derivative overlay (checkbox)")

---
## 9. The Forward Pass — Step by Step

The **forward pass** is the process of computing predictions from input to output.
For each layer $l = 1, \ldots, L$:

1. **Pre-activation:** $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$
2. **Activation:**    $\mathbf{a}^{[l]} = g^{[l]}(\mathbf{z}^{[l]})$

During forward pass we **cache** all intermediate values (z, a per layer) —
they will be needed for backpropagation!

In [ ]:
# ─── Animated Forward Pass ──────────────────────────────────────────────────────
print("=" * 70)
print("THE FORWARD PASS — ANIMATED LAYER-BY-LAYER")
print("=" * 70)

# Build the exact network from slide Practice Problem 1
model_fp = NumpyFNN([2, 2, 1], activations=['relu', 'linear'], init_method='zero')
model_fp.W[0] = np.array([[1.0, -1.0], [0.5,  2.0]])
model_fp.b[0] = np.array([0.0, -1.0])
model_fp.W[1] = np.array([[2.0, -1.0]])
model_fp.b[1] = np.array([0.5])

x_fp = np.array([1.0, 2.0])

print(f"\n📥 Input x = {x_fp}")
print(f"\n{'─'*50}")

for step in forward_pass_step_by_step(x_fp, model_fp):
    l = step['layer']
    act = step['activation']
    print(f"\n🔧 Layer {l} ({act}):")
    print(f"   prev_a  = {step['prev_a']}")
    print(f"   W^[{l}]  =\n{step['W']}")
    print(f"   b^[{l}]  = {step['b']}")
    print(f"   z^[{l}]  = W^[{l}] @ prev_a + b^[{l}] = {step['z']}")
    print(f"   a^[{l}]  = {act}(z) = {step['a']}")

print(f"\n{'─'*50}")
print(f"✅ Final output ŷ = {step['a'][0]:.2f}  (Practice Problem 1 ✓)")

In [ ]:
# ─── Visualised forward pass on beam dataset sample ────────────────────────────
print("\n" + "=" * 55)
print("FORWARD PASS ON BEAM DEFLECTION EXAMPLE")
print("=" * 55)

# Build a trained beam network for demonstration
model_beam_demo = NumpyFNN([6, 16, 8, 1],
                            activations=['relu', 'relu', 'linear'],
                            init_method='he')

# Use just a few samples to demonstrate
X_demo, y_demo = X_beam[:200], y_beam[:200]
scaler_demo = StandardScaler()
X_demo_s = scaler_demo.fit_transform(X_demo)
scaler_y_demo = StandardScaler()
y_demo_s = scaler_y_demo.fit_transform(y_demo.reshape(-1, 1)).ravel()

hist_demo = train_numpy_fnn(model_beam_demo, X_demo_s, y_demo_s,
                             lr=0.005, epochs=300, batch_size=32, verbose=0)

x_single = X_demo_s[0]
print(f"\n📥 Single beam sample: {beam_meta['feature_names']}")
print(f"   Raw values: {X_demo[0]}")

print(f"\n🔄 Forward pass through trained network:")
for step in forward_pass_step_by_step(x_single, model_beam_demo):
    z_abs = np.abs(step['z'])
    a_z = step['a']
    dead = np.mean(a_z == 0) if step['activation'] == 'relu' else 0
    print(f"   Layer {step['layer']} ({step['activation']}): "
          f"shape {step['z'].shape} | |z| ∈ [{z_abs.min():.3f}, {z_abs.max():.3f}] | "
          f"dead: {dead:.0%}")

pred_scaled = model_beam_demo.predict(x_single.reshape(1, -1))[0, 0]
pred_mm = scaler_y_demo.inverse_transform([[pred_scaled]])[0, 0]
true_mm = y_demo[0]
print(f"\n   Prediction: {pred_mm:.2f} mm  |  True: {true_mm:.2f} mm")
print(f"   Error: {abs(pred_mm - true_mm):.2f} mm  (early stage — more training needed!)")

In [ ]:
# ─── Animated forward pass visualisation ─────────────────────────────────────
# Build a 4-layer network for compelling animation
model_anim = NumpyFNN([4, 8, 6, 4, 2], activations=['relu', 'relu', 'relu', 'sigmoid'], init_method='he')
x_anim = np.random.randn(4)

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
input_ax = axes[0]
input_ax.bar(range(4), x_anim, color='steelblue')
input_ax.set_title('Input x\n(Layer 0)', fontsize=10, fontweight='bold')
input_ax.set_xlabel('Neuron index', fontsize=9)
input_ax.set_ylabel('Value', fontsize=9)
input_ax.grid(True, alpha=0.3)

for step in forward_pass_step_by_step(x_anim, model_anim):
    l = step['layer']
    ax = axes[l]
    bar_colors = ['steelblue' if v > 0 else '#9E9E9E' for v in step['a']]
    ax.bar(range(len(step['a'])), step['a'], color=bar_colors)
    ax.set_title(f"a^[{l}]\n{step['activation']}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Neuron index', fontsize=9)
    ax.set_ylabel('Activation', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Layer-by-Layer Activations: Forward Pass\n'
             '(Blue = active, Grey = dead/zero)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Cache stored: z^[1..L], a^[0..L] → needed for backpropagation!")

### 9.1 Vectorised Forward Pass

For a single sample: $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$

For a **batch** of $m$ samples, stack them as rows:
$$\mathbf{Z}^{[l]} = \mathbf{A}^{[l-1]} (\mathbf{W}^{[l]})^T + \mathbf{b}^{[l]}  \quad [\text{broadcasting}]$$

**Vectorised code is 10–100× faster** — modern GPUs are matrix processors!

In [ ]:
# ─── Vectorisation speedup demonstration ──────────────────────────────────────
import time

print("=" * 60)
print("VECTORISED vs LOOP FORWARD PASS — TIMING BENCHMARK")
print("=" * 60)

n_test = 1000
X_timing = np.random.randn(n_test, 6)
W_t = np.random.randn(64, 6)
b_t = np.zeros(64)

# Loop version
t0 = time.perf_counter()
for i in range(n_test):
    z_i = W_t @ X_timing[i] + b_t
    a_i = relu(z_i)
t_loop = time.perf_counter() - t0

# Vectorised version
t0 = time.perf_counter()
Z_batch = X_timing @ W_t.T + b_t
A_batch = relu(Z_batch)
t_vec = time.perf_counter() - t0

speedup = t_loop / max(t_vec, 1e-9)

print(f"\n  {'Method':<20} {'Time':>12} {'Speedup':>10}")
print(f"  {'-'*42}")
print(f"  {'Python loop':<20} {t_loop*1000:>9.2f} ms  {'1×':>10}")
print(f"  {'Vectorised (NumPy)':<20} {t_vec*1000:>9.3f} ms  {speedup:>8.0f}×")

print(f"\n  📐 Shapes verified:")
print(f"     X: {X_timing.shape}, W: {W_t.shape}")
print(f"     Z (vectorised): {Z_batch.shape}")
print(f"     A (vectorised): {A_batch.shape}")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Python Loop\n(1 sample at a time)', 'Vectorised NumPy\n(all samples at once)'],
              [t_loop * 1000, t_vec * 1000],
              color=['tomato', 'steelblue'], edgecolor='black', linewidth=1.5)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title(f'Forward Pass: Loop vs Vectorised\n({n_test} samples, {speedup:.0f}× speedup)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, [t_loop * 1000, t_vec * 1000]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.2f} ms', ha='center', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 10. Loss Functions

The **loss function** $\mathcal{L}$ measures how far our predictions are from the truth.
The choice depends on the task:

| Task | Output Activation | Loss Function |
|------|------------------|---------------|
| Regression | Linear | MSE: $\frac{1}{n}\sum(y - \hat{y})^2$ |
| Binary Classification | Sigmoid | BCE: $-[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| Multi-class | Softmax | CE: $-\sum_k y_k \log \hat{y}_k$ |

In [ ]:
print("=" * 60)
print("LOSS FUNCTIONS — COMPARISON")
print("=" * 60)

# MSE example
y_true_reg = np.array([3.5, 2.0, 7.1, 5.5, 4.0])
y_pred_good = np.array([3.4, 2.1, 7.0, 5.6, 3.9])
y_pred_poor = np.array([1.0, 5.0, 3.0, 8.0, 7.0])

print(f"\n📐 MSE Loss (regression):")
print(f"   Good predictions: MSE = {mse_loss(y_true_reg, y_pred_good):.4f}")
print(f"   Poor predictions: MSE = {mse_loss(y_true_reg, y_pred_poor):.4f}")

# BCE example
y_true_cls = np.array([1, 0, 1, 1, 0], dtype=float)
y_pred_conf = np.array([0.95, 0.05, 0.92, 0.88, 0.07])  # confident & correct
y_pred_wrong = np.array([0.05, 0.92, 0.08, 0.12, 0.91]) # confident & WRONG

print(f"\n📐 Binary Cross-Entropy Loss (classification):")
print(f"   Confident & correct: BCE = {binary_cross_entropy_loss(y_true_cls, y_pred_conf):.4f}")
print(f"   Confident & WRONG:   BCE = {binary_cross_entropy_loss(y_true_cls, y_pred_wrong):.4f}  ← huge penalty!")

# Visualise BCE: penalty for confident wrong predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs = np.linspace(0.01, 0.99, 200)

# BCE for true label = 1
bce_y1 = -np.log(probs)         # y=1: penalise small ŷ
bce_y0 = -np.log(1 - probs)     # y=0: penalise large ŷ

axes[0].plot(probs, bce_y1, 'b-', linewidth=2.5, label='True label = 1: −log(ŷ)')
axes[0].plot(probs, bce_y0, 'r-', linewidth=2.5, label='True label = 0: −log(1−ŷ)')
axes[0].axvline(0.5, color='gray', linewidth=1, linestyle='--', alpha=0.7, label='Decision threshold')
axes[0].set_xlabel('Predicted probability ŷ', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Binary Cross-Entropy Loss\n(Penalises confident wrong predictions)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 5)
axes[0].grid(True, alpha=0.3)

# Loss landscape for a single linear neuron
def single_neuron_loss(w, b, x=2.0, y=5.0):
    y_hat = w * x + b
    return 0.5 * (y_hat - y) ** 2

w_range = np.linspace(-1, 4, 100)
b_range = np.linspace(-3, 6, 100)
fig2, ax_ll = plot_loss_landscape_2d(
    lambda w, b: single_neuron_loss(w, b),
    param_ranges=(w_range, b_range),
    true_params=(1.6, 0.3),
    title="Loss Landscape: Single Neuron\nL = 0.5(wx+b-y)²  for x=2, y=5",
    figsize=(7, 5)
)
plt.tight_layout()
plt.show()
fig2.clear()
plt.close(fig2)

axes[1].scatter(range(len(y_true_reg)), y_true_reg, s=100, c='steelblue', label='True y', zorder=5)
axes[1].scatter(range(len(y_pred_good)), y_pred_good, s=80, c='green', marker='s',
                label='Good pred', zorder=5, alpha=0.8)
axes[1].scatter(range(len(y_pred_poor)), y_pred_poor, s=80, c='tomato', marker='^',
                label='Poor pred', zorder=5, alpha=0.8)
for i in range(len(y_true_reg)):
    axes[1].plot([i, i], [y_true_reg[i], y_pred_good[i]], 'g-', linewidth=1.5, alpha=0.6)
    axes[1].plot([i, i], [y_true_reg[i], y_pred_poor[i]], 'r-', linewidth=1.5, alpha=0.6)
axes[1].set_xlabel('Sample index', fontsize=12)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_title('MSE Loss: Squared vertical residuals', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.figure(num=fig.number)
plt.tight_layout()
plt.show()

# Loss landscape visualisation
fig_ll, ax_ll = plot_loss_landscape_2d(
    lambda w, b: single_neuron_loss(w, b),
    param_ranges=(w_range, b_range),
    true_params=(1.6, 0.3),
    title="Loss Landscape: L = 0.5(wx+b-y)²  for x=2, y=5\n(Optimum = Practice Problem 3 solution!)",
)
plt.tight_layout()
plt.show()

print("\n💡 Cross-entropy penalises CONFIDENT WRONG predictions exponentially more than")
print("   uncertain wrong predictions — this encourages well-calibrated probabilities.")

---
## 11. Training Overview & Gradient Descent

**The 4-step training loop (slides §21):**

1. **Forward pass:** compute $\hat{\mathbf{y}} = f(\mathbf{x}; \theta)$
2. **Loss:** $\mathcal{L} = \ell(\mathbf{y}, \hat{\mathbf{y}})$
3. **Backward pass (backprop):** $\nabla_\theta \mathcal{L}$ via chain rule
4. **Weight update:** $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$

### Practice Problem 3 — One Gradient Step (from slides)

Single linear neuron: $\hat{y} = wx + b$, loss $L = \frac{1}{2}(\hat{y} - y)^2$

Given: $(x, y) = (2, 5)$, $w = 1$, $b = 0$, $\eta = 0.1$

In [ ]:
# ─── Practice Problem 3 — slide-exact gradient step ────────────────────────────
print("=" * 70)
print("PRACTICE PROBLEM 3 — ONE GRADIENT STEP (FROM SLIDES)")
print("=" * 70)

result_p3 = single_neuron_gradient_step(x=2.0, y=5.0, w=1.0, b=0.0, lr=0.1)

print(f"\n📝 Given: x={2.0}, y={5.0}, w={1.0}, b={0.0}, η=0.1")
print(f"\n  Step 1 — Forward pass:")
print(f"    ŷ = w·x + b = 1.0·2 + 0 = {result_p3['y_hat']:.1f}")
print(f"    L = 0.5·(ŷ - y)² = 0.5·({result_p3['y_hat']:.1f} - 5)² = {result_p3['loss']:.1f}")

print(f"\n  Step 2 — Gradients:")
print(f"    ∂L/∂ŷ = ŷ - y = {result_p3['y_hat']:.1f} - 5 = {result_p3['y_hat'] - 5.0:.1f}")
print(f"    ∂L/∂w = (ŷ-y)·x = ({result_p3['y_hat']-5.0:.1f})·2 = {result_p3['grad_w']:.1f}")
print(f"    ∂L/∂b = ŷ-y     = {result_p3['grad_b']:.1f}")

print(f"\n  Step 3 — Update (η=0.1):")
print(f"    w ← 1.0 - 0.1·({result_p3['grad_w']:.1f}) = {result_p3['w_new']:.1f}  ✓")
print(f"    b ← 0.0 - 0.1·({result_p3['grad_b']:.1f}) = {result_p3['b_new']:.1f}  ✓")

print(f"\n  Step 4 — New prediction:")
print(f"    ŷ_new = {result_p3['w_new']:.1f}·2 + {result_p3['b_new']:.1f} = {result_p3['y_hat_new']:.1f} (was {result_p3['y_hat']:.1f})")
print(f"    → Moved CLOSER to y=5 ✓")

In [ ]:
# ─── Animated GD trajectory on the loss landscape ───────────────────────────────
print("\n" + "=" * 55)
print("GRADIENT DESCENT TRAJECTORY — ANIMATED")
print("=" * 55)

# Run 30 gradient steps
history_gd = {
    'w': [1.0], 'b': [0.0], 'loss': []
}
w_curr, b_curr = 1.0, 0.0
lr_gd = 0.1
for step_i in range(30):
    res = single_neuron_gradient_step(2.0, 5.0, w_curr, b_curr, lr_gd)
    w_curr, b_curr = res['w_new'], res['b_new']
    history_gd['w'].append(w_curr)
    history_gd['b'].append(b_curr)
    history_gd['loss'].append(res['loss'])

# Loss landscape + trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

w_range_plot = np.linspace(-1, 5, 100)
b_range_plot = np.linspace(-2, 4, 100)
W_grid, B_grid = np.meshgrid(w_range_plot, b_range_plot)
L_grid = 0.5 * (W_grid * 2 + B_grid - 5) ** 2

ax = axes[0]
cf = ax.contourf(W_grid, B_grid, np.log1p(L_grid), levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax, label='log(1 + Loss)')
ax.plot(history_gd['w'], history_gd['b'], 'w-o', linewidth=2, markersize=5,
        label='GD Path', zorder=5)
ax.scatter(history_gd['w'][0], history_gd['b'][0], c='lime', s=200, zorder=6,
           label='Start', edgecolors='black', linewidths=2)
ax.scatter(history_gd['w'][-1], history_gd['b'][-1], c='red', s=200, zorder=6,
           marker='*', label=f'End (step 30)', edgecolors='black', linewidths=1)
ax.scatter(1.6, 0.3, c='yellow', s=300, zorder=7, marker='*',
           edgecolors='black', linewidths=2, label='Optimum (w*=1.6, b*=0.3)')
ax.set_xlabel('w', fontsize=12)
ax.set_ylabel('b', fontsize=12)
ax.set_title(f'GD Trajectory (η={lr_gd})\nL = 0.5(2w+b-5)²', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')

axes[1].semilogy(history_gd['loss'], 'b-o', linewidth=2, markersize=5)
axes[1].set_xlabel('Gradient Descent Step', fontsize=12)
axes[1].set_ylabel('Loss (log scale)', fontsize=12)
axes[1].set_title('Loss Convergence', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.6, 0.9, f'Final loss: {history_gd["loss"][-1]:.4f}',
             transform=axes[1].transAxes, fontsize=11,
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.suptitle('Practice Problem 3: Gradient Descent on Single Neuron', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Learning rate sensitivity ─────────────────────────────────────────────────
print("\n" + "=" * 55)
print("LEARNING RATE SENSITIVITY")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lrs = [0.01, 0.1, 0.3, 0.6, 1.1]  # last one: diverges!
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(lrs)))

for lr_test, col in zip(lrs, colors):
    w_t, b_t = 1.0, 0.0
    losses_t = []
    ws_t = [w_t]
    for _ in range(30):
        r = single_neuron_gradient_step(2.0, 5.0, w_t, b_t, lr_test)
        w_t, b_t = r['w_new'], r['b_new']
        losses_t.append(r['loss'])
        ws_t.append(w_t)

    label = f'η={lr_test}' + (' 🚨 DIVERGES' if losses_t[-1] > 100 else '')
    axes[0].semilogy(losses_t, '-', color=col, linewidth=2, label=label)
    axes[1].plot(ws_t, '-o', color=col, linewidth=1.5, markersize=4, label=f'η={lr_test}')

axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Loss (log)', fontsize=12)
axes[0].set_title('Convergence for Different Learning Rates', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(1.6, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='w*=1.6 (optimum)')
axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('w (weight value)', fontsize=12)
axes[1].set_title('Weight Trajectory per Learning Rate', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Learning Rate: Too Small = Slow, Too Large = Diverges!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Learning rate widget
# Uncomment in Jupyter/Colab:
# learning_rate_widget()
print("🎮 Run learning_rate_widget() in Jupyter/Colab for interactive exploration!")

In [ ]:
# Training playground widget
# Uncomment in Jupyter/Colab:
# training_playground_widget()
print("🎮 Run training_playground_widget() to adjust LR, batch size, L2, dropout interactively!")

---
## 12. Weight Initialisation

**Why not initialise all weights to zero?**

If $\mathbf{W}^{[l]} = \mathbf{0}$ for all $l$: every neuron in a layer computes the
**same function** (symmetry problem). All gradients are identical → all neurons
updated identically → they STAY identical forever. Depth is useless!

**Solution: random initialisation to break symmetry.**

Two dominant strategies:
- **Xavier / Glorot**: $\mathbf{W} \sim \mathcal{N}(0, \frac{1}{n_{l-1}})$ — for *sigmoid/tanh*
- **He**: $\mathbf{W} \sim \mathcal{N}(0, \frac{2}{n_{l-1}})$ — for *ReLU* (accounts for sparsity)

In [ ]:
print("=" * 70)
print("WEIGHT INITIALISATION — SYMMETRY BREAKING")
print("=" * 70)

# Zero-init demonstration
print("\n❌ Zero Initialisation (symmetry problem):")
model_zero = NumpyFNN([4, 8, 4, 1], activations=['relu','relu','linear'], init_method='zero')
X_sym = np.random.randn(5, 4)
_ = model_zero.forward(X_sym)

for l in range(1, model_zero.n_layers + 1):
    a = model_zero.cache.get(f'a{l}', None)
    if a is not None and l < model_zero.n_layers:
        # All neurons same output?
        all_same = np.allclose(a, a[:, :1])
        print(f"   Layer {l}: all neurons same output? {all_same} {'← Symmetry problem!' if all_same else ''}")

print("\n✅ He Initialisation (symmetry broken):")
model_he = NumpyFNN([4, 8, 4, 1], activations=['relu','relu','linear'], init_method='he')
_ = model_he.forward(X_sym)
for l in range(1, model_he.n_layers + 1):
    a = model_he.cache.get(f'a{l}', None)
    if a is not None and l < model_he.n_layers:
        all_same = np.allclose(a, a[:, :1])
        std_a = np.std(a)
        print(f"   Layer {l}: all same? {all_same}, activation σ = {std_a:.4f}")

In [ ]:
# ─── Visualise activation distributions across init methods ─────────────────────
layer_sizes_init = [10, 32, 32, 32, 1]
fig, axes = plot_init_comparison(
    layer_sizes_init,
    init_methods=['zero', 'random_small', 'xavier', 'he'],
    n_samples=1000, figsize=(16, 9)
)
plt.suptitle('Activation Distributions After Different Weight Initialisations\n'
             'Network: [10, 32, 32, 32, 1] with ReLU, 1000 random inputs',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 Key Takeaways from Initialisation Comparison:")
print("   • Zero init: ZERO variance everywhere — symmetry unbroken, network broken")
print("   • Random small: Low variance — activations die out in deep networks")
print("   • Xavier: Good for sigmoid/tanh but slightly under-scales for ReLU")
print("   • He: Designed for ReLU — maintains healthy variance across all layers")

In [ ]:
# initialization_widget()
print("\n🎮 Run initialization_widget() in Jupyter for interactive exploration!")

---
## 13. Vanishing / Exploding Gradients

**The Vanishing Gradient Problem:**

In deep networks, gradients for early layers become exponentially small through the chain rule:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{[1]}} = \frac{\partial \mathcal{L}}{\partial \mathbf{a}^{[L]}} \cdot \prod_{l=2}^{L} \frac{\partial \mathbf{a}^{[l]}}{\partial \mathbf{a}^{[l-1]}} \cdot \frac{\partial \mathbf{a}^{[1]}}{\partial \mathbf{W}^{[1]}}$$

For **sigmoid**: max derivative = 0.25 → depth-10 network: $0.25^{10} \approx 10^{-6}$ 🔴

For **ReLU**: derivative = 1 for z > 0 → no vanishing (for the active regime) ✅

In [ ]:
print("=" * 70)
print("VANISHING GRADIENT — DEPTH EXPERIMENT")
print("=" * 70)

# Simulate gradient magnitudes through deep sigmoid vs relu networks
def simulate_gradient_norms(activation_name, n_layers=10, n_neurons=32,
                             n_inputs=16, n_trials=5, random_state=42):
    """Simulate mean gradient magnitude at each layer via forward pass variance."""
    rng = np.random.default_rng(random_state)
    all_norms = np.zeros(n_layers)

    for trial in range(n_trials):
        activations_sim = [activation_name] * (n_layers - 1) + ['linear']
        init_m = 'xavier' if activation_name in ('sigmoid', 'tanh') else 'he'
        model_sim = NumpyFNN([n_inputs] + [n_neurons] * (n_layers - 1) + [1],
                              activations=activations_sim, init_method=init_m,
                              random_state=trial)
        X_sim = rng.standard_normal((100, n_inputs))
        _ = model_sim.forward(X_sim)

        # Compute gradient-like quantity: ||∂a/∂z|| at each layer
        deriv_fn = get_derivative(activation_name)
        for l in range(1, n_layers + 1):
            z_l = model_sim.cache.get(f'z{l}', None)
            if z_l is not None:
                grad_norm = np.mean(np.abs(deriv_fn(z_l)))
                all_norms[l - 1] += grad_norm

    return all_norms / n_trials

n_depth = 8
norms_sigmoid = simulate_gradient_norms('sigmoid', n_layers=n_depth)
norms_tanh    = simulate_gradient_norms('tanh',    n_layers=n_depth)
norms_relu    = simulate_gradient_norms('relu',    n_layers=n_depth)

# Cumulative product (gradient flows back through all layers)
cumul_sigmoid = np.cumprod(norms_sigmoid[::-1])[::-1]
cumul_tanh    = np.cumprod(norms_tanh[::-1])[::-1]
cumul_relu    = np.cumprod(norms_relu[::-1])[::-1]

layers = np.arange(1, n_depth + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Per-layer gradient magnitude
axes[0].semilogy(layers, norms_sigmoid, 'r-o', linewidth=2, markersize=8, label='Sigmoid')
axes[0].semilogy(layers, norms_tanh,    'g-s', linewidth=2, markersize=8, label='Tanh')
axes[0].semilogy(layers, norms_relu,    'b-^', linewidth=2, markersize=8, label='ReLU')
axes[0].axhline(0.01, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Danger zone')
axes[0].set_xlabel('Layer', fontsize=12)
axes[0].set_ylabel('Mean |∂a/∂z| (log)', fontsize=12)
axes[0].set_title('Per-Layer Gradient Magnitude', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Cumulative: effective gradient reaching layer 1
axes[1].semilogy(layers, cumul_sigmoid, 'r-o', linewidth=2, markersize=8, label='Sigmoid')
axes[1].semilogy(layers, cumul_tanh,    'g-s', linewidth=2, markersize=8, label='Tanh')
axes[1].semilogy(layers, cumul_relu,    'b-^', linewidth=2, markersize=8, label='ReLU')
axes[1].set_xlabel('Layer (counted from output)', fontsize=12)
axes[1].set_ylabel('Cumulative gradient magnitude (log)', fontsize=12)
axes[1].set_title('Effective Gradient at Layer l\n(Product through chain rule)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Vanishing Gradient Problem — {n_depth}-Layer Network',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📉 Gradient Magnitude Comparison (Layer 1 vs Layer {n_depth}):")
print(f"   Sigmoid: {norms_sigmoid[-1]:.4f} → {norms_sigmoid[0]:.4f}  "
      f"(ratio: {norms_sigmoid[0]/norms_sigmoid[-1]:.2f})")
print(f"   Tanh:    {norms_tanh[-1]:.4f} → {norms_tanh[0]:.4f}  "
      f"(ratio: {norms_tanh[0]/norms_tanh[-1]:.2f})")
print(f"   ReLU:    {norms_relu[-1]:.4f} → {norms_relu[0]:.4f}  "
      f"(ratio: {norms_relu[0]/norms_relu[-1]:.2f})")

print(f"\n💡 Solutions to vanishing gradients:")
print(f"   1. Use ReLU activations (no saturation for z > 0)")
print(f"   2. He initialisation (maintains variance through layers)")
print(f"   3. Batch Normalisation (stabilises activations)")
print(f"   4. Skip connections / ResNets (gradient highway) ← Lecture 11")
print(f"   5. Gradient clipping (for exploding gradients in RNNs) ← Lecture 12")

---
## 14. Regularisation: L2 & Dropout

Neural networks have high capacity → they WILL overfit if unchecked.

**L2 Regularisation (weight decay):**
$$\mathcal{L}_{reg} = \mathcal{L} + \frac{\lambda}{2} \|\mathbf{W}\|^2$$
- Penalises large weights → smoother functions
- Gradient update: $\mathbf{W} \leftarrow (1 - \eta\lambda)\mathbf{W} - \eta \nabla_\mathbf{W}\mathcal{L}$

**Dropout:**
- During training: randomly set each neuron's output to 0 with probability $p$
- Scale by $\frac{1}{1-p}$ to maintain expected value
- During inference: use all neurons (no dropout)
- Forces the network to learn **redundant representations** → generalises better

In [ ]:
print("=" * 70)
print("REGULARISATION — OVERFITTING DEMO")
print("=" * 70)

# Create an overtfit scenario: small dataset, complex patterns
X_sp2, y_sp2 = generate_spiral_dataset(n=80, noise=0.08, n_classes=2)  # small!
X_sp2_n = X_sp2 / (np.std(X_sp2) + 1e-8)

# Split train/val
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_sp2_n, y_sp2.astype(float),
                                             test_size=0.25, random_state=42)

histories_reg = {}
labels = [
    ('No regularisation', 0.0),
    ('L2 λ=0.001', 0.001),
    ('L2 λ=0.01', 0.01),
]

for label, l2 in labels:
    model_r = NumpyFNN([2, 64, 64, 1], activations=['relu', 'relu', 'sigmoid'], init_method='he')
    h = train_numpy_fnn(model_r, X_tr, y_tr,
                         X_val=X_val, y_val=y_val,
                         lr=0.005, epochs=300, batch_size=16,
                         loss_fn='bce', l2_lambda=l2, verbose=0)
    histories_reg[label] = h

fig, axes = plot_regularization_comparison(histories_reg, figsize=(14, 5))
plt.suptitle('Regularisation Effect on Spiral Dataset (Small Training Set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 Regularisation Analysis:")
for label, hist in histories_reg.items():
    if hist['val_loss']:
        gap = hist['val_loss'][-1] - hist['train_loss'][-1]
        print(f"   {label:<25}: final train={hist['train_loss'][-1]:.4f}, "
              f"val={hist['val_loss'][-1]:.4f}, gap={gap:.4f} "
              f"{'🔴 Overfitting' if gap > 0.15 else '🟢 OK'}")

print(f"\n   L2 regularisation REDUCES the generalisation gap by penalising large weights.")
print(f"   Too much L2 (large λ) → underfitting. Use cross-validation to find the sweet spot.")

In [ ]:
# training_playground_widget()
print("\n🎮 Run training_playground_widget() in Jupyter to interactively tune L2 and Dropout!")

---
## 15. Early Stopping

A pragmatic regularisation technique: **stop training when validation loss stops improving**.

This prevents the network from memorising training noise while keeping parameters manageable.

In [ ]:
print("=" * 60)
print("EARLY STOPPING DEMONSTRATION")
print("=" * 60)

# Train for too long — watch validation diverge
model_es = NumpyFNN([2, 64, 64, 1], activations=['relu', 'relu', 'sigmoid'], init_method='he')
hist_es = train_numpy_fnn(model_es, X_tr, y_tr,
                           X_val=X_val, y_val=y_val,
                           lr=0.01, epochs=500, batch_size=16,
                           loss_fn='bce', l2_lambda=0.0, verbose=0)

# Find best val epoch
best_epoch = int(np.argmin(hist_es['val_loss']))
best_val_loss = hist_es['val_loss'][best_epoch]

fig, ax = plt.subplots(figsize=(13, 5))
epochs_es = np.arange(1, len(hist_es['train_loss']) + 1)
ax.plot(epochs_es, hist_es['train_loss'], 'b-', linewidth=2, label='Train Loss')
ax.plot(epochs_es, hist_es['val_loss'], 'r-', linewidth=2, label='Val Loss')
ax.axvline(best_epoch + 1, color='green', linewidth=2.5, linestyle='--',
           label=f'Early Stop @ epoch {best_epoch+1}\nVal Loss = {best_val_loss:.4f}')
ax.fill_between(epochs_es[best_epoch:], 0, 2, alpha=0.1, color='red', label='Overfitting zone')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('BCE Loss', fontsize=12)
ax.set_title('Early Stopping: Stop When Validation Loss Stops Improving',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Best model: epoch {best_epoch+1}, val_loss = {best_val_loss:.4f}")
print(f"   Training beyond this → memorising noise!")

---
## 16. Batch Normalisation

**Batch Normalisation** normalises layer outputs within each mini-batch:

$$\hat{\mathbf{z}} = \frac{\mathbf{z} - \mu_{\text{batch}}}{\sqrt{\sigma_{\text{batch}}^2 + \epsilon}}$$
$$\mathbf{z}_{\text{BN}} = \gamma \hat{\mathbf{z}} + \beta \quad  (\gamma, \beta \text{ learnable})$$

**Benefits:**
1. Stabilises training → allows **higher learning rates**
2. Reduces sensitivity to weight initialisation
3. Acts as mild regularisation (noise from batch statistics)

We compare training with/without BatchNorm using PyTorch.

In [ ]:
print("=" * 65)
print("BATCH NORMALISATION — WITH vs WITHOUT")
print("=" * 65)

if TORCH_AVAILABLE:
    # Build two networks: one with BN, one without
    def make_mlp(use_bn=False):
        layers = []
        dims = [6, 64, 64, 64, 1]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:  # not the last layer
                if use_bn:
                    layers.append(nn.BatchNorm1d(dims[i+1]))
                layers.append(nn.ReLU())
        return nn.Sequential(*layers)

    net_no_bn = make_mlp(use_bn=False)
    net_with_bn = make_mlp(use_bn=True)

    # Prepare beam dataset
    scaler_bn = StandardScaler()
    X_beam_t = scaler_bn.fit_transform(X_beam)
    scaler_y_bn = StandardScaler()
    y_beam_t = scaler_y_bn.fit_transform(y_beam.reshape(-1, 1)).ravel()

    X_tr_b, X_val_b = X_beam_t[:1000], X_beam_t[1000:]
    y_tr_b, y_val_b = y_beam_t[:1000], y_beam_t[1000:]

    X_tr_tensor = torch.FloatTensor(X_tr_b)
    y_tr_tensor = torch.FloatTensor(y_tr_b).unsqueeze(1)
    X_val_tensor = torch.FloatTensor(X_val_b)
    y_val_tensor = torch.FloatTensor(y_val_b).unsqueeze(1)

    ds_tr = TensorDataset(X_tr_tensor, y_tr_tensor)
    ds_val = TensorDataset(X_val_tensor, y_val_tensor)
    dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
    dl_val = DataLoader(ds_val, batch_size=64, shuffle=False)

    criterion = nn.MSELoss()

    # Higher LR with BN — this is the key advantage
    hist_no_bn = train_pytorch_fnn(net_no_bn, dl_tr, criterion,
                                    torch.optim.Adam(net_no_bn.parameters(), lr=0.001),
                                    epochs=60, val_loader=dl_val, verbose=20)

    hist_with_bn = train_pytorch_fnn(net_with_bn, dl_tr, criterion,
                                      torch.optim.Adam(net_with_bn.parameters(), lr=0.005),
                                      epochs=60, val_loader=dl_val, verbose=20)  # 5× LR!

    fig, axes = plot_batch_norm_effect(hist_with_bn, hist_no_bn, figsize=(14, 5))
    plt.suptitle('Batch Normalisation: Higher LR + More Stable Training',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    print(f"\n📊 Final Results:")
    print(f"   Without BN (LR=0.001): val_loss = {hist_no_bn['val_loss'][-1]:.4f}")
    print(f"   With BN    (LR=0.005): val_loss = {hist_with_bn['val_loss'][-1]:.4f}")
    print(f"   → BN allows 5× larger learning rate with MORE stable training!")

else:
    print("⚠️  PyTorch not available — showing batch norm concepts:")
    print("""
    Batch Normalisation in action:
    
    Mini-batch: z = [-2, 1, 3, 0.5, -0.5]
    mean = 0.4, std = 1.71
    
    Normalised: ẑ = (z - 0.4) / 1.71 = [-1.40, 0.35, 1.52, 0.06, -0.53]
    
    Learnable scale + shift: z_BN = γ * ẑ + β
    Initial: γ=1, β=0 → same as ẑ
    
    Benefits: prevents internal covariate shift, stabilises gradients
    """)

---
## 17. CE Application 1 — RC Beam Deflection Prediction

**Full Pipeline: Data → Preprocess → FNN → Train → Evaluate → Interpret**

This is the running example from the slides applied end-to-end.

Features (6): P, L, E, I_gross, M_cr, ρ  
Target: δ (midspan deflection, mm)

In [ ]:
print("=" * 70)
print("CE APP 1 — RC BEAM DEFLECTION PREDICTION")
print("=" * 70)

# ── 1. Data ──────────────────────────────────────────────────────────────────
X_b, y_b, meta_b = generate_beam_deflection_dataset(n=2000, noise_frac=0.05)
X_tr_b, X_val_b, y_tr_b, y_val_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42)

print(f"📊 Dataset: {X_b.shape} samples")
print(f"   Train: {X_tr_b.shape[0]}, Val: {X_val_b.shape[0]}")
print(f"   Target range: [{y_b.min():.1f}, {y_b.max():.1f}] mm")

# ── 2. Preprocess ─────────────────────────────────────────────────────────────
sc_X = StandardScaler()
sc_y = StandardScaler()
X_tr_s  = sc_X.fit_transform(X_tr_b)
X_val_s = sc_X.transform(X_val_b)
y_tr_s  = sc_y.fit_transform(y_tr_b.reshape(-1, 1)).ravel()
y_val_s = sc_y.transform(y_val_b.reshape(-1, 1)).ravel()

# ── 3. NumPy FNN (from-scratch) ───────────────────────────────────────────────
print("\n🔢 NumPy FNN (from scratch):")
model_b_np = NumpyFNN([6, 64, 32, 1], activations=['relu', 'relu', 'linear'], init_method='he')
hist_b_np = train_numpy_fnn(model_b_np, X_tr_s, y_tr_s,
                              X_val=X_val_s, y_val=y_val_s,
                              lr=0.005, epochs=400, batch_size=64, verbose=100)

# ── 4. PyTorch FNN ────────────────────────────────────────────────────────────
if TORCH_AVAILABLE:
    print("\n🔥 PyTorch FNN:")
    net_beam_pt = BeamDeflectionNet(hidden_dims=[64, 32], dropout=0.1)
    params_b = count_torch_parameters(net_beam_pt)
    print(f"   Parameters: {params_b}")

    X_tr_tensor_b = torch.FloatTensor(X_tr_s)
    y_tr_tensor_b = torch.FloatTensor(y_tr_s).unsqueeze(1)
    dl_b_tr = DataLoader(TensorDataset(X_tr_tensor_b, y_tr_tensor_b), batch_size=64, shuffle=True)
    dl_b_val = DataLoader(TensorDataset(torch.FloatTensor(X_val_s),
                                         torch.FloatTensor(y_val_s).unsqueeze(1)), batch_size=64)

    opt_b = torch.optim.Adam(net_beam_pt.parameters(), lr=0.001, weight_decay=1e-4)
    hist_b_pt = train_pytorch_fnn(net_beam_pt, dl_b_tr, nn.MSELoss(), opt_b,
                                   epochs=80, val_loader=dl_b_val, verbose=20)

# ── 5. Evaluate ───────────────────────────────────────────────────────────────
print("\n📈 Evaluation:")

# NumPy model
y_val_pred_s = model_b_np.predict(X_val_s).ravel()
y_val_pred = sc_y.inverse_transform(y_val_pred_s.reshape(-1, 1)).ravel()
rmse_np = np.sqrt(mse_loss(y_val_b, y_val_pred))
r2_np = 1 - np.sum((y_val_b - y_val_pred) ** 2) / np.sum((y_val_b - y_val_b.mean()) ** 2)
print(f"   NumPy FNN:  RMSE = {rmse_np:.2f} mm, R² = {r2_np:.4f}")
print(f"   Linear model RMSE = {np.sqrt(lin_mse):.2f} mm (from §3)")

# ── 6. Visualise ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Training curves
axes[0].semilogy(hist_b_np['train_loss'], 'b-', linewidth=2, label='Train')
axes[0].semilogy(hist_b_np['val_loss'], 'r-', linewidth=2, label='Val')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (scaled)', fontsize=12)
axes[0].set_title('Training Curves', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Predicted vs True
axes[1].scatter(y_val_b, y_val_pred, s=15, alpha=0.5, color='steelblue')
lims = [min(y_val_b.min(), y_val_pred.min()), max(y_val_b.max(), y_val_pred.max())]
axes[1].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('True δ (mm)', fontsize=12)
axes[1].set_ylabel('Predicted δ (mm)', fontsize=12)
axes[1].set_title(f'Pred vs True\nR² = {r2_np:.4f}, RMSE = {rmse_np:.2f} mm',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

# Residuals
residuals = y_val_pred - y_val_b
axes[2].hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].axvline(0, color='red', linewidth=2, linestyle='--')
axes[2].axvline(residuals.mean(), color='orange', linewidth=2,
                label=f'Mean = {residuals.mean():.2f} mm')
axes[2].set_xlabel('Residual (mm)', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('Residual Distribution\n(Should be centred at 0)', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)

plt.suptitle('CE App 1: RC Beam Deflection — FNN vs Linear Model',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ FNN ({r2_np:.4f} R²) vs Linear ({lin_r2:.4f} R²) → {(r2_np/lin_r2 - 1)*100:.0f}% improvement!")
print(f"   FNN captures the piecewise cracking physics — linear model cannot!")

---
## 18. CE Application 2 — Concrete Compressive Strength

**Predicting 28-day compressive strength from mix proportions.**

This is a classic CE regression problem (adapted from the UCI Concrete dataset).

Features (8): cement, water, fine/coarse aggregate, fly ash, slag, superplasticizer, curing time  
Target: f'c (28-day strength, MPa)

In [ ]:
print("=" * 70)
print("CE APP 2 — CONCRETE COMPRESSIVE STRENGTH")
print("=" * 70)

X_c, y_c, meta_c = generate_concrete_strength_dataset(n=1500, noise_std=3.0)
X_tr_c, X_val_c, y_tr_c, y_val_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

print(f"📊 {meta_c['feature_names']}")
print(f"   Samples: {X_c.shape[0]}, f'c range: [{y_c.min():.1f}, {y_c.max():.1f}] MPa")
print(f"   Physics: {meta_c['formula']}")

sc_Xc = StandardScaler()
sc_yc = StandardScaler()
X_tr_cs = sc_Xc.fit_transform(X_tr_c)
X_val_cs = sc_Xc.transform(X_val_c)
y_tr_cs = sc_yc.fit_transform(y_tr_c.reshape(-1, 1)).ravel()
y_val_cs = sc_yc.transform(y_val_c.reshape(-1, 1)).ravel()

model_c = NumpyFNN([8, 64, 32, 16, 1], activations=['relu','relu','relu','linear'], init_method='he')
hist_c = train_numpy_fnn(model_c, X_tr_cs, y_tr_cs,
                          X_val=X_val_cs, y_val=y_val_cs,
                          lr=0.003, epochs=400, batch_size=64, verbose=100)

y_c_pred_s = model_c.predict(X_val_cs).ravel()
y_c_pred = sc_yc.inverse_transform(y_c_pred_s.reshape(-1, 1)).ravel()
rmse_c = np.sqrt(mse_loss(y_val_c, y_c_pred))
r2_c = 1 - np.sum((y_val_c - y_c_pred)**2) / np.sum((y_val_c - y_val_c.mean())**2)

print(f"\n   FNN: RMSE = {rmse_c:.2f} MPa, R² = {r2_c:.4f}")

# Feature sensitivity analysis (simple perturbation)
print("\n🔍 Feature Importance (perturbation sensitivity):")
x_mean = X_val_cs.mean(axis=0)
base_pred = model_c.predict(x_mean.reshape(1, -1))[0, 0]
sensitivities = []
for fi in range(X_c.shape[1]):
    x_perturb = x_mean.copy()
    x_perturb[fi] += 1.0  # +1 std
    new_pred = model_c.predict(x_perturb.reshape(1, -1))[0, 0]
    sensitivities.append(abs(new_pred - base_pred))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
feat_order = np.argsort(sensitivities)[::-1]
axes[0].barh(range(len(sensitivities)),
             [sensitivities[i] for i in feat_order],
             color='steelblue', edgecolor='black', linewidth=0.8)
axes[0].set_yticks(range(len(sensitivities)))
axes[0].set_yticklabels([meta_c['feature_names'][i] for i in feat_order], fontsize=10)
axes[0].set_xlabel('Sensitivity (Δf\'c per +1σ of feature)', fontsize=11)
axes[0].set_title('Feature Importance (Perturbation Analysis)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Pred vs True
axes[1].scatter(y_val_c, y_c_pred, s=15, alpha=0.5, color='steelblue')
lims = [min(y_val_c.min(), y_c_pred.min()), max(y_val_c.max(), y_c_pred.max())]
axes[1].plot(lims, lims, 'r--', linewidth=2)
axes[1].set_xlabel('True f\'c (MPa)', fontsize=12)
axes[1].set_ylabel('Predicted f\'c (MPa)', fontsize=12)
axes[1].set_title(f'Concrete Strength: Pred vs True\nR²={r2_c:.3f}, RMSE={rmse_c:.1f} MPa',
                  fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle("CE App 2: Concrete Compressive Strength Prediction", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

most_important = meta_c['feature_names'][feat_order[0]]
print(f"\n   Most important feature: {most_important}")
print(f"   → Makes physical sense! (w/c ratio controls strength via Abrams' law)")

---
## 19. CE Application 3 — Traffic Flow Classification

**Classify highway traffic conditions from sensor data.**

Classes: Free flow (0), Congested (1), Heavy/stop-go (2)

When should you use a neural network vs. logistic regression?  
→ When the decision boundary is highly nonlinear and you have ≥ 1000 samples.

In [ ]:
print("=" * 70)
print("CE APP 3 — TRAFFIC FLOW CLASSIFICATION")
print("=" * 70)

X_t, y_t, meta_t = generate_traffic_flow_dataset(n=1500, n_classes=3)
X_tr_t, X_val_t, y_tr_t, y_val_t = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

print(f"📊 Features: {meta_t['feature_names']}")
print(f"   Classes: {meta_t['class_names']}")
print(f"   Class distribution: {np.bincount(y_t)}")

sc_Xt = StandardScaler()
X_tr_ts = sc_Xt.fit_transform(X_tr_t)
X_val_ts = sc_Xt.transform(X_val_t)

if TORCH_AVAILABLE:
    net_traffic = TrafficFlowNet(n_classes=3, hidden_dims=[32, 16])
    print(f"\n🔥 Traffic FNN: {count_torch_parameters(net_traffic)}")

    # One-hot encode for CE loss
    y_tr_oh = np.eye(3)[y_tr_t]
    y_val_oh = np.eye(3)[y_val_t]

    dl_t_tr = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr_ts), torch.LongTensor(y_tr_t)),
        batch_size=64, shuffle=True
    )
    dl_t_val = DataLoader(
        TensorDataset(torch.FloatTensor(X_val_ts), torch.LongTensor(y_val_t)),
        batch_size=64
    )

    criterion_ce = nn.CrossEntropyLoss()
    opt_t = torch.optim.Adam(net_traffic.parameters(), lr=0.001)
    hist_t = train_pytorch_fnn(net_traffic, dl_t_tr, criterion_ce, opt_t,
                                epochs=60, val_loader=dl_t_val, verbose=20)

    # Final accuracy
    net_traffic.eval()
    with torch.no_grad():
        val_logits = net_traffic(torch.FloatTensor(X_val_ts))
        y_val_pred_t = val_logits.argmax(dim=1).numpy()
    acc_t = np.mean(y_val_pred_t == y_val_t)

    # Confusion matrix
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(y_val_t, y_val_pred_t)
    print(f"\n   Validation Accuracy: {acc_t:.1%}")
    print(f"\n   Classification Report:")
    print(classification_report(y_val_t, y_val_pred_t,
                                target_names=meta_t['class_names']))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Training curves
    axes[0].plot(hist_t['train_acc'], 'b-', linewidth=2, label='Train Acc')
    axes[0].plot(hist_t['val_acc'], 'r-', linewidth=2, label='Val Acc')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title('Traffic Flow Classification Training', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, alpha=0.3)

    # Confusion matrix
    im = axes[1].imshow(cm, cmap='Blues')
    axes[1].set_xticks(range(3))
    axes[1].set_yticks(range(3))
    axes[1].set_xticklabels(meta_t['class_names'], fontsize=10)
    axes[1].set_yticklabels(meta_t['class_names'], fontsize=10)
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('True', fontsize=12)
    axes[1].set_title(f'Confusion Matrix (Acc={acc_t:.1%})', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=axes[1])
    for i in range(3):
        for j in range(3):
            axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                        fontsize=12, color='white' if cm[i, j] > cm.max()/2 else 'black')

    plt.suptitle("CE App 3: Traffic Flow Condition Classification", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

else:
    print("⚠️ PyTorch not available — showing classification concept:")
    print("   FNN with Softmax output → probability for each of 3 classes")
    print("   Use: argmax(softmax(z^[L])) → predicted class")

---
## 20. Depth × Width Hyperparameter Study

How do we choose `n_hidden_layers` and `n_neurons_per_layer`?

A systematic ablation study on the beam deflection problem.

In [ ]:
print("=" * 65)
print("HYPERPARAMETER STUDY: DEPTH × WIDTH")
print("=" * 65)

depths = [1, 2, 3, 5]
widths = [8, 16, 32, 64]
results_grid = {}

print("\n  Training grid (each for 200 epochs)...")
for depth in depths:
    for width in widths:
        layer_sizes_hw = [6] + [width] * depth + [1]
        m = NumpyFNN(layer_sizes_hw, activations=['relu'] * depth + ['linear'], init_method='he')
        h = train_numpy_fnn(m, X_tr_s, y_tr_s, lr=0.005, epochs=200, batch_size=64, verbose=0)

        y_pred_hw = sc_y.inverse_transform(m.predict(X_val_s).reshape(-1, 1)).ravel()
        rmse_hw = np.sqrt(mse_loss(y_val_b, y_pred_hw))
        results_grid[(depth, width)] = rmse_hw
        print(f"    depth={depth}, width={width}: RMSE={rmse_hw:.2f} mm")

fig, ax = plot_depth_vs_width(results_grid, depths, widths,
                               title="Beam Deflection: RMSE (mm) — Depth × Width Grid")
plt.tight_layout()
plt.show()

best_key = min(results_grid, key=results_grid.get)
print(f"\n✅ Best config: depth={best_key[0]}, width={best_key[1]}, "
      f"RMSE={results_grid[best_key]:.2f} mm")
print(f"\n💡 Practical Rule of Thumb:")
print(f"   • Start simple: 1-2 hidden layers, 32-64 neurons")
print(f"   • Add depth for *very* complex functions, but training gets harder")
print(f"   • Most CE regression tasks are well-served by 2-3 layers")

In [ ]:
# network_builder_widget()
print("\n🎮 Run network_builder_widget() to interactively explore architectures!")

---
## 21. Edge Cases & Stress Testing

A robust notebook must probe the failure modes! We test:
1. **Dead neurons** (ReLU + bad init + large inputs)
2. **Exploding learning rate** (LR too large → divergence)
3. **Tiny dataset** (only 20 samples → severe overfitting)
4. **High-noise data** (signal buried in noise)
5. **Feature scale mismatch** (no normalisation → training failure)

In [ ]:
print("=" * 70)
print("EDGE CASE STRESS TESTS")
print("=" * 70)

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# ── Edge Case 1: Dead neurons ────────────────────────────────────────────────
print("\n🔴 Case 1: Dead Neurons (large-magnitude inputs + ReLU)")
X_dead = np.random.randn(200, 4) * 50  # extreme scale → kills many ReLU neurons
model_dead = NumpyFNN([4, 32, 32, 1], activations=['relu', 'relu', 'linear'], init_method='random_small')
_ = model_dead.forward(X_dead)

dead_fracs = []
for l in range(1, model_dead.n_layers):
    a = model_dead.cache.get(f'a{l}', None)
    if a is not None:
        dead_fracs.append(float(np.mean(a == 0)))

axes[0, 0].bar([f'Layer {l+1}' for l in range(len(dead_fracs))], dead_fracs,
               color=['tomato' if d > 0.5 else 'steelblue' for d in dead_fracs])
axes[0, 0].axhline(0.5, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='50% dead')
axes[0, 0].set_ylabel('Fraction of Dead Neurons', fontsize=11)
axes[0, 0].set_title('Edge Case 1: Dead Neurons\n(Large inputs + ReLU + bad init)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)
for l, d in enumerate(dead_fracs):
    axes[0, 0].text(l, d + 0.02, f'{d:.0%}', ha='center', fontsize=11, fontweight='bold')
print(f"   Dead fraction per layer: {[f'{d:.0%}' for d in dead_fracs]}")
print(f"   → Fix: normalise inputs, use He initialisation!")

# ── Edge Case 2: Exploding LR ────────────────────────────────────────────────
print("\n🔴 Case 2: Exploding Learning Rate")
model_explode = NumpyFNN([2, 16, 1], activations=['relu', 'linear'], init_method='he')
X_exp, y_exp = generate_regression_1d('sin', n=100, noise=0.1)

sc_exp = StandardScaler()
X_exps = sc_exp.fit_transform(X_exp)

losses_by_lr = {}
for lr_exp in [0.001, 0.1, 1.0, 5.0]:
    m_exp = NumpyFNN([1, 16, 1], activations=['relu', 'linear'], init_method='he')
    h_exp = train_numpy_fnn(m_exp, X_exps, y_exp, lr=lr_exp, epochs=50, verbose=0)
    losses_by_lr[lr_exp] = h_exp['train_loss']

colors_exp = ['steelblue', 'green', 'orange', 'tomato']
for (lr_exp, losses), col in zip(losses_by_lr.items(), colors_exp):
    clipped = [min(l, 1e6) for l in losses]
    axes[0, 1].semilogy(clipped, color=col, linewidth=2,
                        label=f'LR={lr_exp}' + (' 🚨' if losses[-1] > 10 else ''))
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Loss (log)', fontsize=11)
axes[0, 1].set_title('Edge Case 2: Learning Rate Too Large\n→ Divergence!', fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)
print(f"   LR=5.0 diverges! Final loss = {losses_by_lr[5.0][-1]:.1e}")
print(f"   → Fix: gradient clipping OR reduce LR OR use adaptive optimiser (Adam)")

# ── Edge Case 3: Tiny dataset ─────────────────────────────────────────────────
print("\n🔴 Case 3: Tiny Dataset (n=20 → severe overfitting)")
X_tiny, y_tiny = generate_regression_1d('sin', n=20, noise=0.2)
X_full, y_full = generate_regression_1d('sin', n=500, noise=0.05)

sc_tiny = StandardScaler()
X_tinys = sc_tiny.fit_transform(X_tiny)
X_fulls = sc_tiny.transform(X_full)

for i, (n_neurons_t, col_t, lw_t) in enumerate([(4, 'steelblue', 2), (32, 'tomato', 2.5), (128, 'purple', 2)]):
    m_tiny = NumpyFNN([1, n_neurons_t, 1], activations=['relu', 'linear'], init_method='he')
    train_numpy_fnn(m_tiny, X_tinys, y_tiny, lr=0.01, epochs=1000, verbose=0)
    y_t_pred = m_tiny.predict(X_fulls).ravel()
    axes[0, 2].plot(X_full.ravel(), y_t_pred, color=col_t, linewidth=lw_t,
                    label=f'{n_neurons_t} neurons')

axes[0, 2].scatter(X_tiny.ravel(), y_tiny, c='black', s=50, zorder=5, label='20 training points')
axes[0, 2].plot(X_full.ravel(), np.sin(2 * np.pi * X_full.ravel()), 'k--',
                linewidth=1.5, alpha=0.6, label='True sin(2πx)')
axes[0, 2].set_xlabel('x', fontsize=11)
axes[0, 2].set_ylabel('y', fontsize=11)
axes[0, 2].set_title('Edge Case 3: Tiny Dataset\n(Big network → overfitting)', fontsize=11, fontweight='bold')
axes[0, 2].legend(fontsize=9)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].set_ylim(-3, 3)
print(f"   Large networks wildly overfit 20 points!")
print(f"   → Fix: more data, regularisation, smaller model, or Gaussian Process instead")

# ── Edge Case 4: High noise ───────────────────────────────────────────────────
print("\n🟡 Case 4: High Noise — Noise Floor Effect")
noise_levels = [0.01, 0.2, 0.5, 1.0]
final_losses_noise = []

for noise_lev in noise_levels:
    X_n, y_n = generate_regression_1d('sin', n=300, noise=noise_lev)
    sc_n = StandardScaler()
    X_ns = sc_n.fit_transform(X_n)
    m_n = NumpyFNN([1, 32, 1], activations=['relu', 'linear'], init_method='he')
    h_n = train_numpy_fnn(m_n, X_ns, y_n, lr=0.005, epochs=300, verbose=0)
    final_losses_noise.append(h_n['train_loss'][-1])

axes[1, 0].bar([str(n) for n in noise_levels], final_losses_noise,
               color=plt.cm.Reds(np.linspace(0.3, 0.9, len(noise_levels))))
axes[1, 0].set_xlabel('Noise Standard Deviation', fontsize=11)
axes[1, 0].set_ylabel('Final Training Loss', fontsize=11)
axes[1, 0].set_title('Edge Case 4: High Noise\n→ Irreducible (Bayes) Error Floor',
                      fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, (noise_lev, loss_n) in enumerate(zip(noise_levels, final_losses_noise)):
    axes[1, 0].text(i, loss_n + 0.01, f'{loss_n:.2f}', ha='center', fontsize=11)
print(f"   Final loss at noise=1.0: {final_losses_noise[-1]:.3f}")
print(f"   → The Bayes error floor is unavoidable — expect ≈ noise²/2 as minimum loss")

# ── Edge Case 5: Feature scale mismatch ──────────────────────────────────────
print("\n🔴 Case 5: Feature Scale Mismatch (no normalisation)")
X_scale = X_beam.copy()  # Raw units span very different ranges!
y_scale_n = y_beam

m_unscaled = NumpyFNN([6, 32, 1], activations=['relu', 'linear'], init_method='he')
m_scaled   = NumpyFNN([6, 32, 1], activations=['relu', 'linear'], init_method='he')

h_unscaled = train_numpy_fnn(m_unscaled, X_scale, y_scale_n, lr=0.001, epochs=200, verbose=0)
sc_X_s5 = StandardScaler()
sc_y_s5 = StandardScaler()
X_s5 = sc_X_s5.fit_transform(X_scale)
y_s5 = sc_y_s5.fit_transform(y_scale_n.reshape(-1, 1)).ravel()
h_scaled = train_numpy_fnn(m_scaled, X_s5, y_s5, lr=0.005, epochs=200, verbose=0)

axes[1, 1].semilogy(h_unscaled['train_loss'], 'tomato', linewidth=2, label='No normalisation 🔴')
axes[1, 1].semilogy(h_scaled['train_loss'], 'steelblue', linewidth=2, label='StandardScaler ✅')
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Loss (log)', fontsize=11)
axes[1, 1].set_title('Edge Case 5: Feature Scale Mismatch\n→ Always normalise inputs!',
                      fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)
print(f"   Without normalisation: {h_unscaled['train_loss'][-1]:.1f}")
print(f"   With normalisation:    {h_scaled['train_loss'][-1]:.5f}")
print(f"   → ALWAYS standardise your inputs before training!")

# ── Edge Case 6: Summary / advice ────────────────────────────────────────────
edge_cases = [
    ('Dead neurons',       'Normalise inputs, He init, Leaky ReLU / ELU'),
    ('Exploding LR',       'Reduce LR, grad clipping, Adam optimiser'),
    ('Tiny dataset',       'More data, regularise (L2/Dropout), simpler model'),
    ('High noise',         'Accept Bayes error, use ensembles, Gaussian Process'),
    ('Scale mismatch',     'ALWAYS use StandardScaler / MinMaxScaler'),
    ('Vanishing grads',    'ReLU, He init, BatchNorm, skip connections'),
]

axes[1, 2].axis('off')
y_pos = 0.95
axes[1, 2].text(0.5, 1.02, '⚠️ Edge Case Summary', transform=axes[1, 2].transAxes,
                ha='center', fontsize=13, fontweight='bold')
for prob, fix in edge_cases:
    axes[1, 2].text(0.05, y_pos, f'❌ {prob}', transform=axes[1, 2].transAxes,
                    fontsize=10, color='tomato', fontweight='bold')
    axes[1, 2].text(0.05, y_pos - 0.06, f'   ✅ {fix}', transform=axes[1, 2].transAxes,
                    fontsize=9, color='steelblue')
    y_pos -= 0.15

plt.suptitle('Edge Case Stress Tests — Know Your Failure Modes!',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 22. Best Practices Summary (matching Slide §36)

| Decision | Recommendation |
|---|---|
| **Architecture** | Start small (1-2 hidden, 32-64 neurons), add depth only if needed |
| **Activation** | ReLU for hidden layers; Sigmoid/Softmax for classification output |
| **Initialisation** | He for ReLU; Xavier for Sigmoid/Tanh |
| **Learning rate** | Start at 1e-3 (Adam) or 1e-2 (SGD); use LR schedulers |
| **Batch size** | 32-256; larger = more stable but less regularisation effect |
| **Regularisation** | L2 + Dropout (0.2-0.5) for small datasets |
| **BatchNorm** | Use when depth > 4 or training is unstable |
| **Optimiser** | Adam for most CE tasks; SGD+momentum for large-scale training |
| **Preprocessing** | Always StandardScaler or MinMaxScaler on inputs AND outputs |
| **When NOT to use NNs** | < 500 samples → Gaussian Process / linear regression instead |

In [ ]:
print("=" * 65)
print("BEST PRACTICES SUMMARY — DECISION CHECKLIST")
print("=" * 65)

checklist = [
    ("Dataset size",        "< 500", "Use GP, Ridge, or SVR instead of FNN"),
    ("Dataset size",        "500 – 5000", "Simple FNN (1-2 layers), strong regularisation"),
    ("Dataset size",        "> 5000", "Deeper FNN (2-4 layers), mild regularisation"),
    ("Input features",      "Any", "ALWAYS StandardScaler before training"),
    ("Hidden activation",   "Default", "ReLU (fast, no vanishing grad in positive regime)"),
    ("Output activation",   "Regression", "Linear (none)"),
    ("Output activation",   "Binary cls", "Sigmoid → BCE loss"),
    ("Output activation",   "Multi-class", "Softmax → Cross-Entropy loss"),
    ("Initialisation",      "With ReLU", "He Normal"),
    ("Initialisation",      "With Sigmoid/Tanh", "Xavier / Glorot"),
    ("Learning rate",       "Adam optimiser", "1e-3 default; halve if training diverges"),
    ("Deep network",        "> 4 layers", "Add BatchNorm between layers"),
    ("Overfitting",         "val > train loss", "L2 (λ=1e-4 to 1e-2) + Dropout (0.2)"),
    ("Slow training",       "val plateau", "Reduce LR by 10× or use cosine annealing"),
]

print(f"\n  {'Situation':<22} {'Condition':<25} {'Recommendation'}")
print(f"  {'-'*75}")
for situation, condition, recommendation in checklist:
    print(f"  {situation:<22} {condition:<25} {recommendation}")

In [ ]:
print("\n" + "=" * 65)
print("WHEN TO PREFER FNNs vs. SIMPLER MODELS")
print("=" * 65)

comparison_table = {
    "Method": ["Linear Regression", "Decision Trees/RF", "SVM", "FNN"],
    "n_samples": ["> 50", "> 200", "> 500", "> 1000"],
    "Interpretable": ["✅ Yes", "🟡 Partial", "🔴 No", "🔴 No"],
    "Nonlinear": ["🔴 No", "✅ Yes", "✅ Yes (kernel)", "✅ Yes"],
    "Feature engineering": ["Needed", "Minimal", "Moderate", "Minimal"],
    "CE use cases": ["Simple correlations", "Tabular CE data", "Classification", "Complex physics"],
}

for key, values in comparison_table.items():
    print(f"\n  {key:<22}: {' | '.join(str(v)[:18].ljust(18) for v in values)}")

---
## 23. Practice Problems

### Problem 1 — Forward Pass by Hand (Slides §31)

Given network with:
- Input: **x = [1, 2]**
- Layer 1: **W¹ = [[1, -1], [0.5, 2]]**, **b¹ = [0, -1]**, activation = ReLU
- Layer 2: **W² = [[2, -1]]**, **b² = [0.5]**, activation = Linear

**Task:** Compute z¹, a¹, z², ŷ

### Problem 3 — Gradient Step by Hand (Slides §33)

Single neuron: ŷ = wx + b, L = 0.5(ŷ-y)²

Given: x=2, y=5, w=1, b=0, η=0.1

**Task:** Compute ŷ, L, ∂L/∂w, ∂L/∂b, w_new, b_new

In [ ]:
print("=" * 70)
print("PRACTICE PROBLEMS — CODE SOLUTIONS")
print("=" * 70)

# ── Problem 1 — Forward Pass ────────────────────────────────────────────────
print("\n📝 Problem 1 — Forward Pass:")
model_pp1 = NumpyFNN([2, 2, 1], activations=['relu', 'linear'], init_method='zero')
model_pp1.W[0] = np.array([[1.0, -1.0], [0.5,  2.0]])
model_pp1.b[0] = np.array([0.0, -1.0])
model_pp1.W[1] = np.array([[2.0, -1.0]])
model_pp1.b[1] = np.array([0.5])

x_pp = np.array([1.0, 2.0])
_ = model_pp1.forward(x_pp.reshape(1, -1))

z1 = model_pp1.cache['z1'].ravel()
a1 = model_pp1.cache['a1'].ravel()
z2 = model_pp1.cache['z2'].ravel()
a2 = model_pp1.cache['a2'].ravel()

print(f"\n  z¹ = W¹x + b¹ = {z1}")
print(f"  a¹ = ReLU(z¹)  = {a1}")
print(f"  z² = W²a¹ + b² = {z2}")
print(f"  ŷ = z²         = {a2}")

# Check with manual computation
z1_manual = np.array([1*1 + (-1)*2 + 0,  0.5*1 + 2*2 + (-1)])
a1_manual = relu(z1_manual)
z2_manual = np.array([2*a1_manual[0] + (-1)*a1_manual[1] + 0.5])

print(f"\n  Manual verification:")
print(f"  z¹ = [1·1+(-1)·2+0, 0.5·1+2·2+(-1)] = {z1_manual}")
print(f"  a¹ = ReLU({z1_manual}) = {a1_manual}")
print(f"  z² = 2·{a1_manual[0]}+(-1)·{a1_manual[1]}+0.5 = {z2_manual}")
print(f"  ✅ Match: {np.allclose(z2, z2_manual)}")

# ── Problem 3 — Gradient Step ───────────────────────────────────────────────
print("\n" + "─" * 50)
print("📝 Problem 3 — One Gradient Step:")
res_pp3 = single_neuron_gradient_step(x=2.0, y=5.0, w=1.0, b=0.0, lr=0.1)
print(f"\n  ŷ = w·x+b  = 1·2+0 = {res_pp3['y_hat']:.1f}")
print(f"  L = 0.5·(ŷ-y)² = 0.5·(2-5)² = {res_pp3['loss']:.1f}")
print(f"  ∂L/∂w = (ŷ-y)·x = -3·2 = {res_pp3['grad_w']:.1f}")
print(f"  ∂L/∂b = (ŷ-y)   = -3   = {res_pp3['grad_b']:.1f}")
print(f"  w_new = 1 - 0.1·(-6) = {res_pp3['w_new']:.1f} ✓")
print(f"  b_new = 0 - 0.1·(-3) = {res_pp3['b_new']:.1f} ✓")
print(f"  ŷ_new = 1.6·2 + 0.3  = {res_pp3['y_hat_new']:.1f}  (closer to y=5 ✓)")

# ── Practice extension: run more steps ───────────────────────────────────────
print("\n  Extension: Running 10 more gradient steps:")
w_pp, b_pp = res_pp3['w_new'], res_pp3['b_new']
for step_i in range(10):
    r = single_neuron_gradient_step(2.0, 5.0, w_pp, b_pp, lr=0.1)
    w_pp, b_pp = r['w_new'], r['b_new']
    if step_i % 3 == 0 or step_i == 9:
        print(f"    Step {step_i+2}: ŷ={r['y_hat']:.3f}, L={r['loss']:.4f}")
print(f"  w* ≈ {w_pp:.3f}, b* ≈ {b_pp:.3f}  (true optimum: w*=1.6, b*≈0.3 for this single point)")

---
## 24. Summary & Key Takeaways

### What We Learned

1. **Motivation**: Linear models fail for nonlinear CE physics (piecewise cracking, Abrams' law)

2. **Universal Approximation**: A single hidden layer can approximate any continuous function —
   but depth + width + training ease all matter in practice

3. **Architecture**: Stack of (Linear → Activation) blocks; dimensions track as
   $\mathbf{W}^{[l]} \in \mathbb{R}^{n_l \times n_{l-1}}$

4. **Why Nonlinearity**: Linear stacks collapse; ReLU is the default choice

5. **Forward Pass**: Cache all z, a for later backprop; vectorise over batches

6. **Loss**: MSE for regression, BCE/CE for classification

7. **Training**: 4-step loop (forward → loss → backward → update); learning rate is critical

8. **Initialisation**: He for ReLU, Xavier for sigmoid/tanh; NEVER zero-init for hidden layers

9. **Vanishing Gradients**: Use ReLU + He init + BatchNorm; avoid deep sigmoid networks

10. **Regularisation**: L2 (weight decay) + Dropout + Early Stopping

11. **CE Applications**:
    - Beam deflection: FNN learns piecewise cracking (R² >> linear)
    - Concrete strength: 8-feature regression with Abrams' physics baseline
    - Traffic classification: multiclass softmax with 3 conditions

12. **Next Steps**: CNNs for image-based CE tasks (crack detection, remote sensing) → Lecture 11

In [ ]:
print("=" * 70)
print("LECTURE 10 SUMMARY — FEEDFORWARD NEURAL NETWORKS")
print("=" * 70)

summary_stats = {
    "Beam deflection R² (FNN)":    f"{r2_np:.4f}",
    "Concrete strength RMSE":      f"{rmse_c:.2f} MPa",
    "Traffic classification acc":  f"{acc_t:.1%}" if TORCH_AVAILABLE else "N/A (PyTorch needed)",
    "Linear model R² (beam)":      f"{lin_r2:.4f}",
    "FNN improvement over linear": f"{(r2_np/lin_r2 - 1)*100:.0f}%",
}

print("\n📊 Results Summary:")
for k, v in summary_stats.items():
    print(f"   {k:<40}: {v}")

print("\n🔑 The 3 Most Important Lessons:")
print("   1. ALWAYS normalise your inputs (StandardScaler)")
print("   2. Use ReLU + He init as the default setup")
print("   3. Start with a small network and use regularisation — less is more!")

print("\n📚 Further Reading:")
print("   • Goodfellow et al. (2016): Deep Learning, Chapters 6-8")
print("   • He et al. (2015): Delving deep into rectifiers (He initialisation)")
print("   • Ioffe & Szegedy (2015): Batch Normalisation")
print("   • Hinton et al. (2012): Improving neural networks by preventing co-adaptation (Dropout)")
print("   • Géron (2022): Hands-On ML with Scikit-Learn, Keras & TensorFlow, Ch 10-11")

print("\n🚀 Coming Up: Lecture 11 — Convolutional Neural Networks for Image-Based CE Tasks")
print("   (crack detection, land use classification, pavement distress assessment)")